# TPC raw ADC → IBF and primary-density maps

This notebook keeps the original full-resolution native **2D** histograms as the spatial input:

`h_adc_pad_vs_layer_side{side}_sec{sector:02d}_R{module}_{QA_NAME}`

and uses the global **3D** `#phi × R × time` histograms only to measure the accepted-time fraction:

`h_adc_phi_vs_radius_vs_tbin_side{side}_{QA_NAME}`

The time selection is therefore applied offline without requiring the sector TH3 to have native pad/layer granularity. The accepted window is `[TIME_START, TIME_END)`, where:

- `TIME_START = 0` by default;
- if `APPLY_TIME_CUT = True`, `TIME_START = TIME_CUT_START_TBIN` (for example 270);
- `TIME_END` is found independently for every input ROOT file from the global time distribution,
  at the sustained transition from collision signal to the late-time noise floor;
- `TIME_END_TBIN_OVERRIDE` can replace the automatic endpoint for validation.

Every native ADC map is normalized by

\[
N_{\rm events}\,(TIME_END-TIME_START),
\]

so different runs are compared as ADC per frame per recorded raw TPC time bin before the
pad-area and gain corrections are applied. The detected exposure and time profile are saved
to the output ROOT files for auditing.

## Part I — IBF
1. Measure `f_accept(#phi,R) = ADC(accepted time) / ADC(all stored time)` from the global TH3.
2. Multiply each full-resolution native `pad × layer` TH2 bin by the corresponding `f_accept` value.
3. Normalize by event count × accepted record length.
3. Detect vertical dead FEE regions, radial dead stripes, blocks, and isolated outliers.
4. Repair only **inside each sector**; sector gaps are never bridged.
5. Convert cleaned ADC to an areal-density proxy using `r*dphi*dr`.
6. Smoothly extrapolate R1 inward to the 21 cm inner field cage.
7. Build side-0 and side-1 `rho(r,phi)` maps and diagnostic 2D fits.

## Part II — primary density
Starts again from the cleaned, exposure-normalized ADC map, optionally divides by a gain
map, and makes the same `r`, `phi`, and 2D-fit diagnostics.


In [1]:
import ROOT as root
import numpy as np
import math
import os
import subprocess
from array import array
from pathlib import Path
from scipy import ndimage

root.gROOT.SetBatch(True)
root.TH1.AddDirectory(False)

# ------------------------------------------------------------------
# ROOT presentation style
# ------------------------------------------------------------------
root.gStyle.SetOptStat(0)
root.gStyle.SetOptFit(0)
root.gStyle.SetTitleFontSize(0.055)
root.gStyle.SetLabelSize(0.045,"XYZ")
root.gStyle.SetTitleSize(0.052,"XYZ")
root.gStyle.SetTitleOffset(1.05,"X")
root.gStyle.SetTitleOffset(1.25,"Y")
root.gStyle.SetLegendTextSize(0.035)
ROOT_COLORS=[root.kBlue+1,root.kRed+1,root.kGreen+2,root.kMagenta+1,
             root.kOrange+7,root.kCyan+2,root.kViolet+1,root.kGray+2]
ROOT_QA_OBJECTS={}

# ============================================================
# USER CONFIGURATION
# ============================================================

INPUT_ROOT = os.environ.get("TPC_DENSITY_INPUT", "input/qa2/QA2_79513.root")
RUN_TAG = os.environ.get("TPC_DENSITY_TAG", "")
IS_BATCH_CHILD = os.environ.get("TPC_DENSITY_BATCH_CHILD", "0") == "1"
QA_NAME = "PHGarfieldRawHitsQA"
# Event count is read from h_nEvents_<QA_NAME>. Set a positive value only to override it.
N_EVENTS_OVERRIDE = None

# ------------------------------------------------------------------
# OFFLINE TIME WINDOW / RUN EXPOSURE
# ------------------------------------------------------------------
# Default: no early-time/crossing-0 removal. Set True to remove the first
# TIME_CUT_START_TBIN raw time bins (270 is the intended crossing-0 option).
APPLY_TIME_CUT = True
TIME_CUT_START_TBIN = 260

# Normally leave this None. A number forces the record end [start,end).
TIME_END_TBIN_OVERRIDE = None

# Automatic record-end detection uses the global high-ADC hit-time profile when
# available (less sensitive to low-level noise), otherwise the global ADC profile.
TIME_PROFILE_PREFER_HIGH_ADC = True

# How the coarse timing information is mapped onto the native fine TH2.
# "side" (default) preserves the old spatial map exactly and applies only one
# accepted/all ADC factor per TPC side. "layer" allows only radial dependence;
# "local" reproduces the previous phi x R correction and is kept for QA only.
TIME_ACCEPTANCE_SPATIAL_MODE = "side"   # "side", "layer", or "local"
TIME_END_SEARCH_MIN_TBIN = 400
TIME_END_SMOOTH_BINS = 3
TIME_END_PRE_BINS = 5
TIME_END_CONFIRM_BINS = 4
TIME_END_DROP_FRACTION = 0.35
TIME_END_NOISE_NSIGMA = 6.0
TIME_END_MIN_SIGNAL_FRACTION = 0.01
TIME_END_TAIL_FRACTION = 0.25
TPC_TIMEBIN_NS = 53.326

OUTPUT_DIR = Path(os.environ.get("TPC_DENSITY_OUTPUT_DIR", "output"))
OUTPUT_SUFFIX = f"_{RUN_TAG}" if RUN_TAG else ""
QA_PLOT_DIR = OUTPUT_DIR / f"qa_plots{OUTPUT_SUFFIX}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QA_PLOT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_ROOT = OUTPUT_DIR / f"TPC_IBF_primary_maps{OUTPUT_SUFFIX}.root"
QA_ROOT = OUTPUT_DIR / f"TPC_IBF_primary_QA{OUTPUT_SUFFIX}.root"

N_SIDES = 2
N_SECTORS = 12
N_MODULES = 3
LAYERS_PER_MODULE = 16
N_PADS = [94, 128, 192]

MIN_RADII_MODULE_MM = [314.9835971194212, 416.5920253621078, 589.1096334932404]
DELTA_RADII_MODULE_MM = [5.657908935192669, 10.206891263554285, 10.970474549405283]
AVR_DPHI_MODULE = [0.005307305707805798, 0.003959159565794228, 0.0026514278029726376]
AVR_MINPHI_MODULE = [1.3219316149976348, 1.3179398673058638, 1.3175593270455]

USE_IDEAL_PAD_MAP = True
IDEAL_PAD_MAP_LIBRARY = "libtpctrackreco.so"

IFC_RADIUS_CM = 21.0

# ------------------------------------------------------------------
# DEAD / HOT CHANNEL AND FEE / STRIPE DETECTION
# ------------------------------------------------------------------
DEAD_CELL_REL_THRESHOLD = 0.18

# FEE = vertical multi-pad region. Small alive islands inside a dead FEE are
# allowed; only genuinely dead-like cells in the FEE region are reconstructed.
MIN_DEAD_FEE_WIDTH = 2
DEAD_FEE_LAYER_FRACTION = 0.45
FEE_BRIDGE_PADS = 2
FEE_CONTEXT_PADS = 12

# Robust dead-FEE reconstruction: estimate the two healthy boundaries for the
# complete 16-layer module, smooth those boundary curves in radius, and only
# then interpolate across the missing pad region. This avoids layer-by-layer
# noise being imprinted into a reconstructed FEE.
FEE_MIN_CONTEXT_CELLS = 4
FEE_CONTEXT_MEDIAN_LAYERS = 3
FEE_CONTEXT_GAUSS_SIGMA_LAYERS = 0.8

# Dead stripes are detector/readout structure and are treated separately:
# R1/R2: normally full-sector layer-wide stripe
# R3: allow half-sector stripe as well as full-sector stripe
STRIPE_REL_THRESHOLD = 0.28
FULL_STRIPE_FRACTION = 0.70
HALF_STRIPE_FRACTION = 0.70
OTHER_HALF_MAX_DEAD_FRACTION = 0.35
STRIPE_NEIGHBOR_LAYERS = 3
ALLOW_R3_HALF_STRIPES = True

# Larger irregular dead blocks
MIN_DEAD_BLOCK_WIDTH = 2
MIN_DEAD_BLOCK_LAYERS = 2
MIN_DEAD_BLOCK_CELLS = 6
BLOCK_CONTEXT_PADS = 12
BLOCK_CONTEXT_LAYERS = 3

# Single-cell/pad outliers. This operates only inside each native sector, so
# physical sector boundaries are never crossed.
REPAIR_SINGLE_OUTLIERS = True
SINGLE_DEAD_FACTOR = 0.25
SINGLE_HOT_FACTOR = 2.5
SINGLE_OUTLIER_PAD_RADIUS = 2
SINGLE_OUTLIER_LAYER_RADIUS = 1
SINGLE_OUTLIER_MIN_NEIGHBORS = 5
SINGLE_OUTLIER_PASSES = 2

# Final robust physical smoothing, still sector-local. A local trimmed mean/median
# suppresses small fluctuations but preserves broad GEM structures.
SMOOTH_NATIVE_MAPS = True
SMOOTH_PAD_RADIUS = 2
SMOOTH_LAYER_RADIUS = 1
SMOOTH_BLEND = 0.75

# IBF keeps the measured dead stripes. Underlying clean maps used for
# extrapolation and the default primary map reconstruct those stripes first.
KEEP_DEAD_STRIPES_IN_IBF = True
SAVE_PRIMARY_WITH_DEAD_STRIPES = True

# ------------------------------------------------------------------
# PAD/SAMPA ESTIMATOR
# ------------------------------------------------------------------
# "raw", "sum", "mean", "median", "trimmed_mean"
ADC_ESTIMATOR = "trimmed_mean"
TRIM_FRACTION = 0.15
CLIP_OUTLIERS = True
CLIP_NSIGMA = 5.0

GLOBAL_PHI_BINS = 720
PHI_MIN = -math.pi
PHI_MAX = math.pi

# ------------------------------------------------------------------
# SAFE POWER-LAW EXTRAPOLATION TO 21 cm
# rho(r,phi) = A(phi)/r^alpha
# Used independently for IBF and primary.
# ------------------------------------------------------------------
ALPHA_INITIAL = 2.2
ALPHA_BOUNDS = (0.8, 3.7)
N_R1_FIT_LAYERS = 12
N_INNER_EXTRAP_BINS = 24

ALPHA_LOSS = "soft_l1"
ALPHA_F_SCALE = 1.0
MIN_PHI_BINS_PER_R_LAYER = 30
MIN_HEALTHY_FRACTION_FOR_ALPHA = 0.35

N_R1_AMPLITUDE_LAYERS = 16
MIN_POINTS_FOR_PHI_AMPLITUDE = 12
MIN_R1_COVERAGE_LAYERS = 2

# ---------------------------------------------------------------
# phi dependence of the INNER extrapolation
# ---------------------------------------------------------------
# IMPORTANT: the model is parameterized at a reference radius:
#
#   rho(r,phi) = rho_ref(phi) * (R_REF / r)^alpha(phi)
#
# This keeps rho_ref(phi) in density units even when alpha varies with phi.
# The old A(phi)/r^alpha parameterization made A change units with alpha and
# could create artificial factor-of-10 structure.
R_REF_CM = None                  # strongly recommended: first measured R1 radius

# alpha(phi) is intentionally extracted with MUCH coarser phi binning than
# the final density map. 36 bins = 10 degrees/bin = 3 bins/sector.
ALPHA_N_PHI_BINS = 4
ALPHA_LOW_HARMONIC = 0

# Three alpha QA models:
#   constant     : one alpha for the whole side
#   coarse       : coarse robust alpha(phi), interpolated to fine phi
#   lowharmonic  : robust n<=4 fit to the coarse values
ALPHA_PHI_FINAL = "lowharmonic"
MIN_R1_POINTS_PER_ALPHA_BIN = 8

# rho_ref(phi) / multiplicity models:
#   binwise       : robust value in every fine phi bin
#   sector_smooth : robust smoothing INSIDE each physical sector only
#   highharmonic  : optional global high-order Fourier comparison
RHOREF_PHI_FINAL = "sector_smooth"
RHOREF_SECTOR_MEDIAN_WINDOW = 21
RHOREF_SECTOR_GAUSS_SIGMA = 3.0

# After requiring >=10 R1 measurements per fine phi bin, some bins can still be
# missing because pad centers move slightly with radius. Fill those missing values
# only by interpolation INSIDE the same physical sector.
RHOREF_FILL_MISSING_WITHIN_SECTOR = True
RHOREF_HIGH_HARMONIC = 24

# ------------------------------------------------------------------
# Radial damping of phi modulation below the first measured R1 layer
# ------------------------------------------------------------------
# rho_ref(phi) is decomposed into mean + modulation. The modulation can
# gradually fade toward the IFC while the radial 1/r^alpha behavior is kept.
#
# Values below mean the fraction of phi modulation retained at r=IFC:
#   1.0 -> preserve full measured phi structure all the way inward
#   0.5 -> keep half of it at the IFC
#   0.0 -> become phi-flat at the IFC
PHI_MODULATION_IFC_FRACTIONS = [1.0, 0.5, 0.0]
PHI_MODULATION_IFC_FINAL = 0.5

# "linear" gives S(r) = f_IFC + (1-f_IFC)*(r-IFC)/(R_first-IFC)
PHI_MODULATION_DAMPING = "linear"

ALLOW_EXTERNAL_R_REF = False

# ------------------------------------------------------------------
# Smooth diagnostic 2D fit
# ------------------------------------------------------------------
FOURIER_ORDER = 12
RADIAL_POLY_ORDER = 3
FIT_MIN_DENSITY = 0.0
ROBUST_2D_FIT = True
ROBUST_2D_LOSS = "soft_l1"

# ------------------------------------------------------------------
# Gain map / primary density
# ------------------------------------------------------------------
GAIN_MAP_FILE = "../distort/input/layer_gain_79513_Mariia_side01.root"

GAIN_MAP_HIST_SIDE0 = "hGainMap_side0_South"
GAIN_MAP_HIST_SIDE1 = "hGainMap_side1_North"

# Gain histogram axes from the supplied ROOT file:
# x = sector 0..11, y = TPC layer index 0..47.
# Raw-hit TPC layers are 7..54.
GAIN_LAYER_OFFSET = 7

PRIMARY_DIVIDE_BY_GAIN = True

# ------------------------------------------------------------------
# ROBUST CROSS-RUN ADC RATIO -> k_eff CALIBRATION
# ------------------------------------------------------------------
# k_eff(run,side) = KEFF_REFERENCE_VALUE * fitted_ADC_ratio(run/reference,side).
# The fit uses only measured TPC bins that are healthy in BOTH runs, rejects
# residual ratio outliers, forms one robust ratio per radial layer, then fits
# those layer ratios with a constant over the measured TPC.
KEFF_REFERENCE_RUN = 79513
KEFF_REFERENCE_VALUE = 1.55
KEFF_STABLE_R_MIN_CM = 31.0
KEFF_STABLE_R_MAX_CM = 76.0
KEFF_STABLE_MAX_UNSTABLE_FRACTION = 0.05
KEFF_STABLE_MIN_SIGNAL_FRACTION = 0.25
KEFF_STABLE_RATIO_CLIP_NSIGMA = 4.0
KEFF_STABLE_LAYER_CLIP_NSIGMA = 3.5
KEFF_STABLE_MIN_PHI_FRACTION = 0.20
KEFF_STABLE_MIN_LAYERS = 12
KEFF_STABLE_REL_ERROR_FLOOR = 0.005

# Cross-run display profiles use a robust estimator rather than ROOT projections
# (which sum every phi/r bin and are much more sensitive to dead readout regions).
COMPARISON_PROFILE_ESTIMATOR = "median"

# ------------------------------------------------------------------
# OPTIONAL MULTI-FILE BATCH
# ------------------------------------------------------------------
# Leave empty for normal interactive use. To process several runs, put the
# input ROOT files here and call run_batch() in the final notebook cell.

BATCH_INPUT_ROOTS = [#QA_79514.root  QA_79515.root  QA_79516.root  QA_79526.root  QA_79528.root  QA_79529.root
    "input/qa2/QA2_79513.root",
    "input/qa2/QA2_79508.root",
    "input/qa2/QA2_79509.root",
    "input/qa2/QA2_79510.root",
    "input/qa2/QA2_79511.root",
    "input/qa2/QA2_79512.root",
    "input/qa2/QA2_79514.root",
    "input/qa2/QA2_79515.root",
    "input/qa2/QA2_79516.root",
    "input/qa2/QA2_79523.root",
    "input/qa2/QA2_79528.root",
    "input/qa2/QA2_79529.root",
    "input/qa2/QA2_81558.root",
    #"input/qa2/QA2_76905.root"
]
NOTEBOOK_FILE = Path(os.environ.get("TPC_DENSITY_NOTEBOOK", "TPC_IBF_PrimaryDensity_Extraction_v19_STABLE_SPATIAL_PIPELINE.ipynb"))
RUN_COMPARISON_AFTER_BATCH = True

COMPARISON_RUNS = list(dict.fromkeys(int(Path(f).stem.split("_")[-1]) for f in BATCH_INPUT_ROOTS))

print("Configuration loaded")
print("Input:", INPUT_ROOT)
print("Gain map:", GAIN_MAP_FILE)
print("Output ROOT:", OUTPUT_ROOT)
print("QA ROOT:", QA_ROOT)


Welcome to JupyROOT 6.30/06
Configuration loaded
Input: input/qa2/QA2_79513.root
Gain map: ../distort/input/layer_gain_79513_Mariia_side01.root
Output ROOT: output/TPC_IBF_primary_maps.root
QA ROOT: output/TPC_IBF_primary_QA.root


### Stability restoration in v19

This version restores the v13 spatial behavior. The fine native TH2 is no longer multiplied by a noisy local $R\times\phi$ timing fraction before defect finding. By default, timing contributes only a per-side accepted/all ADC factor plus the run-specific `N_frames × accepted_time` exposure. The old layer-by-layer dead-FEE interpolation and ROOT summed comparison profiles are restored. `TIME_ACCEPTANCE_SPATIAL_MODE='local'` remains available only as a diagnostic.


In [2]:
%jsroot on

## Geometry and ROOT helpers

Dead-region finding is performed in native sector coordinates. The conversion to
global `(r,phi)` happens only after repair.

This version first tries to load the same `IdealPadMap` used by
`PHGarfieldRawHitsQA`. If available, it uses exact `get_radius(layer)` and
`get_phi(side,sector,layer,local_pad)` values. This is especially important for
the sector that crosses `phi = +/-pi`: its pad centers are unwrapped only while
calculating `dphi`, then wrapped back to `[-pi,pi)` for histogram filling.

The supplied average geometry is retained only as a fallback.


In [3]:
def get_hist_name(side, sector, module):
    return f"h_adc_pad_vs_layer_side{side}_sec{sector:02d}_R{module+1}_{QA_NAME}"

def get_hist3_name(side, sector, module):
    return f"h_adc_pad_vs_layer_vs_tbin_side{side}_sec{sector:02d}_R{module+1}_{QA_NAME}"

def get_global_adc_time_name(side):
    return f"h_adc_phi_vs_radius_vs_tbin_side{side}_{QA_NAME}"

def get_nevents_name():
    return f"h_nEvents_{QA_NAME}"

def th2_to_numpy(h):
    nx, ny = h.GetNbinsX(), h.GetNbinsY()
    z = np.empty((ny, nx), dtype=float)
    for iy in range(ny):
        for ix in range(nx):
            z[iy, ix] = h.GetBinContent(ix+1, iy+1)
    x = np.array([h.GetXaxis().GetBinCenter(i+1) for i in range(nx)])
    y = np.array([h.GetYaxis().GetBinCenter(i+1) for i in range(ny)])
    return z, x, y

# ------------------------------------------------------------
# Exact IdealPadMap geometry when available
# ------------------------------------------------------------
ideal_pad_map = None

if USE_IDEAL_PAD_MAP:
    try:
        root.gSystem.Load(IDEAL_PAD_MAP_LIBRARY)
        if hasattr(root, "IdealPadMap"):
            ideal_pad_map = root.IdealPadMap()
            if (not ideal_pad_map.is_loaded()) and ideal_pad_map.load_from_cdb(0) != 0:
                print("WARNING: IdealPadMap CDB load failed; using fallback geometry")
                ideal_pad_map = None
        else:
            print("WARNING: ROOT.IdealPadMap is unavailable; using fallback geometry")
    except Exception as exc:
        print("WARNING: could not initialize IdealPadMap:", exc)
        ideal_pad_map = None

print("Geometry source:", "IdealPadMap" if ideal_pad_map is not None else "fallback averages")

def global_layer(module, local_layer):
    return 7 + 16*module + local_layer

def module_layer_radii_cm(module):
    if ideal_pad_map is not None:
        return np.array([
            float(ideal_pad_map.get_radius(global_layer(module, il)))
            for il in range(LAYERS_PER_MODULE)
        ])
    r0 = MIN_RADII_MODULE_MM[module] / 10.0
    dr = DELTA_RADII_MODULE_MM[module] / 10.0
    return r0 + dr*(np.arange(LAYERS_PER_MODULE)+0.5)

def module_layer_dr_cm(module):
    r = module_layer_radii_cm(module)
    # Voronoi-like radial width around layer centers, bounded by midpoint.
    e = np.empty(len(r)+1)
    e[1:-1] = 0.5*(r[:-1]+r[1:])
    e[0] = r[0] - 0.5*(r[1]-r[0])
    e[-1] = r[-1] + 0.5*(r[-1]-r[-2])
    return np.diff(e)

def wrap_phi(phi):
    return (phi + np.pi) % (2*np.pi) - np.pi

def sector_phi_centers(module, sector, local_layer, side=0):
    npads = N_PADS[module]

    if ideal_pad_map is not None:
        layer = global_layer(module, local_layer)
        vals = [
            float(ideal_pad_map.get_phi(side, sector, layer, local_pad))
            for local_pad in range(npads)
        ]
        return np.array([wrap_phi(v) for v in vals], dtype=float)

    dphi = AVR_DPHI_MODULE[module]
    sector_pitch = 2*np.pi/12.0
    phi0 = AVR_MINPHI_MODULE[module] + sector*sector_pitch
    return wrap_phi(phi0 + (np.arange(npads)+0.5)*dphi)

def layer_dphi(module, local_layer, side=0, sector=0):
    ph = sector_phi_centers(module, sector, local_layer, side)
    # unwrap before taking differences; this is important for the sector
    # that crosses +/-pi.
    pu = np.unwrap(ph)
    if len(pu) < 2:
        return AVR_DPHI_MODULE[module]
    return float(np.median(np.abs(np.diff(pu))))

def pad_area_cm2(module, local_layer, side=0):
    r = module_layer_radii_cm(module)[local_layer]
    dr = module_layer_dr_cm(module)[local_layer]
    dphi = layer_dphi(module, local_layer, side)
    return r * dphi * dr

def robust_scale(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return 0.0
    med = np.median(x)
    mad = np.median(np.abs(x-med))
    return 1.4826*mad if mad > 0 else np.std(x)

def clipped_values(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return x
    med = np.median(x)
    sig = robust_scale(x)
    if sig <= 0 or not np.isfinite(sig):
        return x
    return x[np.abs(x-med) <= CLIP_NSIGMA*sig]

def estimate_values(x, mode=None):
    if mode is None:
        mode = ADC_ESTIMATOR
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan

    if mode == "sum":
        return np.sum(x)
    if mode == "raw":
        return np.mean(x)

    if CLIP_OUTLIERS:
        xc = clipped_values(x)
        if xc.size:
            x = xc

    if mode == "mean":
        return np.mean(x)
    if mode == "median":
        return np.median(x)
    if mode == "trimmed_mean":
        xs = np.sort(x)
        ntrim = int(TRIM_FRACTION*len(xs))
        if 2*ntrim >= len(xs):
            return np.median(xs)
        return np.mean(xs[ntrim:len(xs)-ntrim])

    raise ValueError(mode)

def weighted_median(values, weights):
    values = np.asarray(values, float)
    weights = np.asarray(weights, float)
    good = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not np.any(good):
        return np.nan
    v = values[good]
    w = weights[good]
    order = np.argsort(v)
    v = v[order]
    w = w[order]
    c = np.cumsum(w)
    return v[np.searchsorted(c, 0.5*c[-1])]


Geometry source: fallback averages


Error in <TUnixSystem::FindDynamicLibrary>: libtpctrackreco.so does not exist in /home/yoren/bnl/ROOT/install/lib:.:/home/yoren/bnl/ROOT/install/lib:/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v3:/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v2:/lib/x86_64-linux-gnu/tls/haswell/x86_64:/lib/x86_64-linux-gnu/tls/haswell:/lib/x86_64-linux-gnu/tls/x86_64:/lib/x86_64-linux-gnu/tls:/lib/x86_64-linux-gnu/haswell/x86_64:/lib/x86_64-linux-gnu/haswell:/lib/x86_64-linux-gnu/x86_64:/lib/x86_64-linux-gnu:/usr/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v3:/usr/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v2:/usr/lib/x86_64-linux-gnu/tls/haswell/x86_64:/usr/lib/x86_64-linux-gnu/tls/haswell:/usr/lib/x86_64-linux-gnu/tls/x86_64:/usr/lib/x86_64-linux-gnu/tls:/usr/lib/x86_64-linux-gnu/haswell/x86_64:/usr/lib/x86_64-linux-gnu/haswell:/usr/lib/x86_64-linux-gnu/x86_64:/usr/lib/x86_64-linux-gnu:/lib/glibc-hwcaps/x86-64-v3:/lib/glibc-hwcaps/x86-64-v2:/lib/tls/haswell/x86_64:/lib/tls/haswell:/lib/tls/x86_64:/lib/tls:/lib/

# Part I — IBF source shape

## 1. Load native sector/module maps

In [4]:
fin = root.TFile.Open(INPUT_ROOT, "READ")
if not fin or fin.IsZombie():
    raise OSError(f"Cannot open {INPUT_ROOT}")

def read_event_count(fin):
    h=fin.Get(get_nevents_name())
    if N_EVENTS_OVERRIDE is not None:
        n=float(N_EVENTS_OVERRIDE)
        if not np.isfinite(n) or n<=0: raise ValueError("N_EVENTS_OVERRIDE must be positive")
        return n
    if not h:
        raise KeyError(f"Missing {get_nevents_name()}; set N_EVENTS_OVERRIDE only for legacy files")
    n=float(h.GetBinContent(1))
    if not np.isfinite(n) or n<=0:
        raise ValueError(f"Invalid event count {n} in {get_nevents_name()}")
    return n

def projection_z_arrays(h3):
    p=h3.ProjectionZ(f"pz_{h3.GetName()}_{root.TUUID().AsString()}",1,h3.GetNbinsX(),1,h3.GetNbinsY(),"e")
    p.SetDirectory(0)
    n=p.GetNbinsX()
    y=np.array([p.GetBinContent(i+1) for i in range(n)],float)
    c=np.array([p.GetXaxis().GetBinCenter(i+1) for i in range(n)],float)
    lo=np.array([p.GetXaxis().GetBinLowEdge(i+1) for i in range(n)],float)
    hi=np.array([p.GetXaxis().GetBinUpEdge(i+1) for i in range(n)],float)
    return y,c,lo,hi

def find_highadc_time_hist(fin,side):
    suffix=f"_phi_vs_radius_vs_tbin_side{side}_{QA_NAME}"
    names=[k.GetName() for k in fin.GetListOfKeys()
           if k.GetName().startswith("h_nhits_adc_gt") and k.GetName().endswith(suffix)]
    if not names: return None
    if len(names)>1:
        print(f"WARNING: multiple high-ADC time histograms for side {side}: {names}; using {names[0]}")
    return fin.Get(names[0])

def combine_time_hists(hists):
    profiles=[]; centers=lows=highs=None
    for h in hists:
        y,c,lo,hi=projection_z_arrays(h)
        if centers is None:
            centers,lows,highs=c,lo,hi
        elif len(c)!=len(centers) or not np.allclose(c,centers):
            raise ValueError("Side-0 and side-1 time axes do not match")
        profiles.append(y)
    return np.sum(profiles,axis=0),centers,lows,highs

def combined_time_profile(fin):
    # Never mix unlike quantities between sides. Use high-ADC hit counts for
    # both sides only when both histograms exist and contain signal; otherwise
    # use ADC for both sides.
    if TIME_PROFILE_PREFER_HIGH_ADC:
        hs=[find_highadc_time_hist(fin,side) for side in range(N_SIDES)]
        if all(hs):
            y,c,lo,hi=combine_time_hists(hs)
            if np.nanmax(y)>0:
                return y,c,lo,hi,[h.GetName() for h in hs]

    hs=[fin.Get(get_global_adc_time_name(side)) for side in range(N_SIDES)]
    if not all(hs):
        missing=[get_global_adc_time_name(side) for side,h in enumerate(hs) if not h]
        raise KeyError("Missing global ADC time histograms: "+", ".join(missing))
    y,c,lo,hi=combine_time_hists(hs)
    return y,c,lo,hi,[h.GetName() for h in hs]

def smooth_time_profile(y):
    y=np.asarray(y,float)
    y=np.nan_to_num(y,nan=0.0,posinf=0.0,neginf=0.0)
    size=max(1,int(TIME_END_SMOOTH_BINS))
    if size%2==0: size+=1
    s=ndimage.median_filter(y,size=size,mode="nearest") if size>1 else y.copy()
    return ndimage.gaussian_filter1d(s,0.8,mode="nearest") if len(s)>2 else s

def detect_record_end(centers,lows,highs,y):
    """Return the exclusive raw-tbin end where signal makes a sustained drop to noise."""
    centers=np.asarray(centers,float); y=np.asarray(y,float)
    if len(y)<8 or np.nanmax(y)<=0: raise ValueError("Time profile has no usable signal")
    sm=smooth_time_profile(y)

    ntail=max(8,int(math.ceil(TIME_END_TAIL_FRACTION*len(sm))))
    tail=sm[-ntail:]
    noise=float(np.median(tail))
    mad=float(1.4826*np.median(np.abs(tail-noise)))
    peak=float(np.nanpercentile(sm,95))
    threshold=noise+max(TIME_END_NOISE_NSIGMA*mad,
                        TIME_END_MIN_SIGNAL_FRACTION*max(peak-noise,0.0),
                        1e-12)

    search=int(np.searchsorted(centers,float(TIME_END_SEARCH_MIN_TBIN),side="left"))
    pre=max(2,int(TIME_END_PRE_BINS)); confirm=max(2,int(TIME_END_CONFIRM_BINS))
    candidate=None

    # Primary criterion: abrupt local drop from the decreasing signal into a
    # sustained noise-level region. This avoids treating the last isolated
    # noise spike as recorded signal.
    for i in range(max(search,pre),len(sm)-2*confirm+1):
        before=float(np.median(sm[i-pre:i]))
        after=float(np.median(sm[i:i+confirm]))
        future=float(np.median(sm[i:i+2*confirm]))
        if before<=threshold: continue
        if after<=threshold and after<=TIME_END_DROP_FRACTION*before and future<=1.5*threshold:
            candidate=i
            break

    # Fallback: last sustained above-noise bin, ignoring isolated late spikes.
    if candidate is None:
        active=sm>threshold
        last=None
        for i in range(search,len(sm)):
            lo=max(search,i-confirm+1)
            if np.count_nonzero(active[lo:i+1])>=max(1,confirm-1):
                last=i
        if last is None:
            raise RuntimeError("Could not determine the recorded time endpoint")
        candidate=min(last+1,len(sm))

    # The smoothing used above can shift a sharp edge by one coarse time bin.
    # Refine locally on the unsmoothed profile: choose the first bin that itself
    # is at noise level and is followed by a sustained noise-level window.
    for j in range(max(search,candidate-3),min(len(y),candidate+4)):
        before=float(np.median(y[max(0,j-pre):j])) if j>0 else 0.0
        after=float(np.median(y[j:min(len(y),j+confirm)]))
        if before>threshold and y[j]<=1.5*threshold and after<=1.5*threshold:
            candidate=j
            break

    if candidate>=len(sm):
        end=float(highs[-1]+0.5)
    else:
        end=float(lows[candidate]+0.5)
    end=max(end,1.0)
    return int(round(end)),sm,{"noise":noise,"noise_sigma":mad,"threshold":threshold,"peak":peak,
                              "candidate_index":int(candidate)}

N_EVENTS=read_event_count(fin)
time_profile,time_centers,time_lows,time_highs,time_profile_sources=combined_time_profile(fin)
auto_end_tbin,time_profile_smooth,time_end_diag=detect_record_end(
    time_centers,time_lows,time_highs,time_profile)

RECORD_END_TBIN=int(TIME_END_TBIN_OVERRIDE) if TIME_END_TBIN_OVERRIDE is not None else auto_end_tbin
REQUESTED_TIME_START_TBIN=int(TIME_CUT_START_TBIN) if APPLY_TIME_CUT else 0
if RECORD_END_TBIN<=REQUESTED_TIME_START_TBIN:
    raise ValueError(f"Invalid time window [{REQUESTED_TIME_START_TBIN},{RECORD_END_TBIN})")

def selected_zbins(h3,start_tbin,end_tbin):
    z=h3.GetZaxis()
    centers=np.array([z.GetBinCenter(i) for i in range(1,z.GetNbins()+1)],float)
    use=np.where((centers>=start_tbin)&(centers<end_tbin))[0]
    if not len(use):
        raise ValueError(f"No TH3 time bins in requested interval [{start_tbin},{end_tbin})")
    first,last=int(use[0]+1),int(use[-1]+1)
    # Convert ROOT edges (-0.5 convention) back to raw integer-tbin boundaries.
    eff_start=int(round(z.GetBinLowEdge(first)+0.5))
    eff_end=int(round(z.GetBinUpEdge(last)+0.5))
    return first,last,eff_start,eff_end

# Use the global phi x R x time histogram to snap the requested cut to the
# actual coarse time bins. The per-sector TH3 may be spatially rebinned; that
# no longer matters because the native full-resolution TH2 is kept as the
# spatial source of truth.
reference_h3=fin.Get(get_global_adc_time_name(0)) or fin.Get(get_global_adc_time_name(1))
if not reference_h3:
    raise KeyError("No global phi x R x time histogram found")

TIME_ZBIN_FIRST,TIME_ZBIN_LAST,TIME_START_TBIN,TIME_END_TBIN=selected_zbins(
    reference_h3,REQUESTED_TIME_START_TBIN,RECORD_END_TBIN)
RECORD_LENGTH_TBINS=TIME_END_TBIN-TIME_START_TBIN
EXPOSURE_FRAMES_TBINS=N_EVENTS*RECORD_LENGTH_TBINS
EXPOSURE_FRAMES_NS=EXPOSURE_FRAMES_TBINS*TPC_TIMEBIN_NS

print("Time profile source:",", ".join(time_profile_sources))
print(f"Events/frames: {N_EVENTS:g}")
print(f"Auto record end: {auto_end_tbin} raw tbins; selected end: {RECORD_END_TBIN}")
print(f"Accepted time window: [{TIME_START_TBIN},{TIME_END_TBIN}) = {RECORD_LENGTH_TBINS} raw tbins")
if REQUESTED_TIME_START_TBIN!=TIME_START_TBIN:
    print(f"NOTE: requested start {REQUESTED_TIME_START_TBIN} snapped to TH3 bin boundary {TIME_START_TBIN}")
print(f"Exposure: {EXPOSURE_FRAMES_TBINS:g} frame*tbins ({EXPOSURE_FRAMES_NS:g} frame*ns)")
print("Endpoint diagnostics:",time_end_diag)


def project_xy_time_range(h3, first, last, name):
    z = h3.GetZaxis()
    old_first, old_last = z.GetFirst(), z.GetLast()
    z.SetRange(first, last)
    h = h3.Project3D("xy")
    z.SetRange(old_first, old_last)
    h.SetName(name)
    h.SetDirectory(0)
    return h


def global_time_acceptance(fin,side):
    """Return accepted/all ADC fraction on the global (R,phi) grid for one side.

    The denominator covers every stored TH3 time bin because the native fine TH2
    also contains every stored time bin. Thus multiplying the TH2 by this ratio
    removes both the optional early-time interval and the late post-record tail.
    """
    h3=fin.Get(get_global_adc_time_name(side))
    if not h3:
        raise KeyError(f"Missing {get_global_adc_time_name(side)}")

    first,last,eff_start,eff_end=selected_zbins(h3,TIME_START_TBIN,TIME_END_TBIN)
    if (eff_start,eff_end)!=(TIME_START_TBIN,TIME_END_TBIN):
        raise ValueError(f"Inconsistent global time axis side {side}: [{eff_start},{eff_end})")

    hacc=project_xy_time_range(h3,first,last,f"h_time_accept_adc_side{side}_{root.TUUID().AsString()}")
    hall=project_xy_time_range(h3,1,h3.GetNbinsZ(),f"h_time_all_adc_side{side}_{root.TUUID().AsString()}")
    acc,phi_centers,r_centers=th2_to_numpy(hacc)
    total,phi2,r2=th2_to_numpy(hall)
    if acc.shape!=total.shape or not np.allclose(phi_centers,phi2) or not np.allclose(r_centers,r2):
        raise ValueError(f"Global time projections have inconsistent axes on side {side}")

    side_ratio=float(np.sum(acc)/np.sum(total)) if np.sum(total)>0 else 1.0
    layer_den=np.sum(total,axis=1)
    layer_num=np.sum(acc,axis=1)
    layer_ratio=np.divide(layer_num,layer_den,out=np.full_like(layer_num,side_ratio),where=layer_den>0)

    frac=np.divide(acc,total,out=np.full_like(acc,np.nan),where=total>0)
    # Empty global cells should not destroy a non-empty fine TH2 bin. Fall back
    # first to the same radial layer's global fraction, then to the whole side.
    for ir in range(frac.shape[0]):
        bad=~np.isfinite(frac[ir])
        if np.any(bad): frac[ir,bad]=layer_ratio[ir] if np.isfinite(layer_ratio[ir]) else side_ratio
    frac=np.clip(np.nan_to_num(frac,nan=side_ratio,posinf=side_ratio,neginf=0.0),0.0,1.0)

    phi_edges=np.array([h3.GetXaxis().GetBinLowEdge(i) for i in range(1,h3.GetNbinsX()+1)] +
                       [h3.GetXaxis().GetBinUpEdge(h3.GetNbinsX())],float)
    local_valid=frac[np.isfinite(frac)]
    local_rms=float(np.std(local_valid)) if len(local_valid) else 0.0
    return {"fraction":frac,"accepted":acc,"all":total,"phi_edges":phi_edges,
            "r_centers":r_centers,"haccepted":hacc,"hall":hall,
            "side_ratio":side_ratio,"layer_ratio":layer_ratio,"local_rms":local_rms}


time_acceptance_global={side:global_time_acceptance(fin,side) for side in range(N_SIDES)}
for side,item in time_acceptance_global.items():
    print(f"Side {side}: global ADC accepted/all = {item['side_ratio']:.5f}; "
          f"local fraction RMS = {item['local_rms']:.5f}")
print("Timing spatial correction mode:", TIME_ACCEPTANCE_SPATIAL_MODE)


def fine_acceptance_map(side,module,sector,shape):
    """
    Timing acceptance mapped to the native pad/layer map.

    Default "side" mode intentionally preserves the old spatial pattern:
    timing determines the accepted ADC fraction and exposure, but cannot inject
    noisy phi-by-phi structure into defect finding or the final density map.
    """
    ny,nx=shape
    if ny!=LAYERS_PER_MODULE or nx!=N_PADS[module]:
        raise ValueError(f"Unexpected native map shape side={side} R{module+1} sec={sector}: {shape}")
    info=time_acceptance_global[side]
    mode=str(TIME_ACCEPTANCE_SPATIAL_MODE).lower()

    if mode=="side":
        return np.full(shape,info["side_ratio"],float)

    if mode=="layer":
        out=np.empty(shape,float)
        for ilayer in range(ny):
            ir=global_layer(module,ilayer)-7
            out[ilayer,:]=info["layer_ratio"][ir]
        return out

    if mode=="local":
        frac_global=info["fraction"]
        phi_edges=info["phi_edges"]
        out=np.empty(shape,float)
        for ilayer in range(ny):
            ir=global_layer(module,ilayer)-7
            if ir<0 or ir>=frac_global.shape[0]:
                raise IndexError(f"Global layer index {ir} outside timing map")
            phis=sector_phi_centers(module,sector,ilayer,side)
            iphi=np.searchsorted(phi_edges,phis,side="right")-1
            iphi=np.clip(iphi,0,frac_global.shape[1]-1)
            out[ilayer,:]=frac_global[ir,iphi]
        return out

    raise ValueError(f"Unknown TIME_ACCEPTANCE_SPATIAL_MODE={TIME_ACCEPTANCE_SPATIAL_MODE!r}")


raw_maps={}
missing=[]

for side in range(N_SIDES):
    for module in range(N_MODULES):
        for sector in range(N_SECTORS):
            name2=get_hist_name(side,sector,module)
            h2=fin.Get(name2)
            if not h2:
                missing.append(name2); continue

            z,pads,layers=th2_to_numpy(h2)
            faccept=fine_acceptance_map(side,module,sector,z.shape)
            accepted_adc=z*faccept
            raw_maps[(side,module,sector)]={
                "adc":accepted_adc/EXPOSURE_FRAMES_TBINS,
                "pads":pads,
                "layers":layers,
                "time_acceptance":faccept,
                "name":name2,
            }

print(f"Loaded {len(raw_maps)} full-resolution TH2 maps; timing correction mode={TIME_ACCEPTANCE_SPATIAL_MODE}")
if missing:
    print(f"Missing {len(missing)} histograms")
    print("\n".join(missing[:10]))


Time profile source: h_nhits_adc_gt100_phi_vs_radius_vs_tbin_side0_PHGarfieldRawHitsQA, h_nhits_adc_gt100_phi_vs_radius_vs_tbin_side1_PHGarfieldRawHitsQA
Events/frames: 993
Auto record end: 970 raw tbins; selected end: 970
Accepted time window: [260,970) = 710 raw tbins
Exposure: 705030 frame*tbins (3.75964e+07 frame*ns)
Endpoint diagnostics: {'noise': 0.0, 'noise_sigma': 0.0, 'threshold': 5293.7379941636445, 'peak': 529373.7994163644, 'candidate_index': 97}
Side 0: global ADC accepted/all = 0.64784; local fraction RMS = 0.07899
Side 1: global ADC accepted/all = 0.64808; local fraction RMS = 0.08057
Timing spatial correction mode: side
Loaded 72 full-resolution TH2 maps; timing correction mode=side


## 2. Automatic defect masks

The native sector maps are classified before any smoothing:

- **dead FEE:** vertical multi-pad regions, allowing small live islands inside;
- **dead stripe:** a radial layer that is strongly suppressed relative to its neighboring layers;
- **dead block:** larger irregular connected dead regions;
- **single hot/dead cells:** handled later by a local robust outlier pass.

Dead stripes are special. They are reconstructed for the underlying physical source
used by extrapolation and primary-density extraction, but they are restored to their
measured dead values in the final IBF map.


In [5]:
def contiguous_true_regions(mask):
    mask=np.asarray(mask,bool)
    out=[]; start=None
    for i,v in enumerate(mask):
        if v and start is None: start=i
        elif not v and start is not None:
            out.append((start,i)); start=None
    if start is not None: out.append((start,len(mask)))
    return out

def robust_dead_cell_candidates(adc):
    adc=np.asarray(adc,float)
    pos=np.where(np.isfinite(adc) & (adc>0),adc,np.nan)
    row=np.nanmedian(pos,axis=1); col=np.nanmedian(pos,axis=0); glob=np.nanmedian(pos)
    row=np.where(np.isfinite(row),row,glob); col=np.where(np.isfinite(col),col,glob)
    expected=np.maximum(row[:,None],col[None,:])
    candidate=(~np.isfinite(adc)) | (np.isfinite(expected) & (expected>0) &
                                    (adc<DEAD_CELL_REL_THRESHOLD*expected))
    return candidate,expected

def radial_dead_fraction(adc,il):
    """Dead fraction of one layer relative to nearby radial layers, pad-by-pad."""
    rows=[j for d in range(1,STRIPE_NEIGHBOR_LAYERS+1)
          for j in (il-d,il+d) if 0<=j<adc.shape[0]]
    if not rows: return np.zeros(adc.shape[1],bool)
    ref=np.nanmedian(np.where(np.asarray(adc[rows])>0,adc[rows],np.nan),axis=0)
    return (~np.isfinite(adc[il])) | (np.isfinite(ref) & (ref>0) &
                                      (adc[il]<STRIPE_REL_THRESHOLD*ref))

def detect_dead_structures(adc,module):
    candidate,expected=robust_dead_cell_candidates(adc)
    nlayer,npad=adc.shape; mid=npad//2

    # FEE columns: bridge one/two alive columns inside an otherwise dead vertical FEE.
    colfrac=np.mean(candidate,axis=0)
    fee_cols=colfrac>=DEAD_FEE_LAYER_FRACTION
    if FEE_BRIDGE_PADS>0:
        fee_cols=ndimage.binary_closing(fee_cols,structure=np.ones(2*FEE_BRIDGE_PADS+1))
    fee_regions=[x for x in contiguous_true_regions(fee_cols) if x[1]-x[0]>=MIN_DEAD_FEE_WIDTH]
    fee_region_mask=np.zeros_like(candidate)
    for lo,hi in fee_regions: fee_region_mask[:,lo:hi]=True

    # Only low cells inside the region are repaired; live islands are retained.
    fee_mask=fee_region_mask & candidate

    stripe_full=np.zeros_like(candidate)
    stripe_half=np.zeros_like(candidate)
    stripe_details=[]

    for il in range(nlayer):
        dead=radial_dead_fraction(adc,il)
        usable=~fee_region_mask[il]
        if np.count_nonzero(usable)<4: continue

        frac=np.count_nonzero(dead & usable)/np.count_nonzero(usable)
        if frac>=FULL_STRIPE_FRACTION:
            stripe_full[il,usable]=True
            stripe_details.append(("full",il,float(frac)))
            continue

        if module==2 and ALLOW_R3_HALF_STRIPES:
            left=usable[:mid]; right=usable[mid:]
            lf=np.count_nonzero(dead[:mid] & left)/max(np.count_nonzero(left),1)
            rf=np.count_nonzero(dead[mid:] & right)/max(np.count_nonzero(right),1)
            if lf>=HALF_STRIPE_FRACTION and rf<=OTHER_HALF_MAX_DEAD_FRACTION:
                stripe_half[il,:mid]=left
                stripe_details.append(("half-left",il,float(lf)))
            elif rf>=HALF_STRIPE_FRACTION and lf<=OTHER_HALF_MAX_DEAD_FRACTION:
                stripe_half[il,mid:]=right
                stripe_details.append(("half-right",il,float(rf)))

    # Irregular dead blocks, excluding already-classified FEE/stripe structure.
    already=fee_mask | stripe_full | stripe_half
    remaining=candidate & ~already
    labels,nlab=ndimage.label(remaining,structure=np.ones((3,3),int))
    block=np.zeros_like(candidate); details=[]
    for lab in range(1,nlab+1):
        yy,xx=np.where(labels==lab)
        if not len(xx): continue
        width=xx.max()-xx.min()+1; height=yy.max()-yy.min()+1
        if width>=MIN_DEAD_BLOCK_WIDTH and height>=MIN_DEAD_BLOCK_LAYERS and len(xx)>=MIN_DEAD_BLOCK_CELLS:
            block[yy,xx]=True
            details.append((int(yy.min()),int(yy.max()+1),int(xx.min()),int(xx.max()+1),int(len(xx))))

    return {"candidate":candidate,"expected":expected,"fee":fee_mask,
            "fee_regions":fee_regions,"fee_region_mask":fee_region_mask,
            "stripe_full":stripe_full,"stripe_half":stripe_half,
            "stripe":stripe_full|stripe_half,"stripe_details":stripe_details,
            "block":block,"block_details":details,"col_dead_fraction":colfrac}

auto_masks={key:detect_dead_structures(item["adc"],key[1]) for key,item in raw_maps.items()}

print("Dead-FEE regions:",sum(len(v["fee_regions"]) for v in auto_masks.values()))
print("Dead stripes:",sum(len(v["stripe_details"]) for v in auto_masks.values()))
print("Large dead blocks:",sum(len(v["block_details"]) for v in auto_masks.values()))


/tmp/ipykernel_10402/1002375550.py:14: RuntimeWarning: All-NaN slice encountered
  row=np.nanmedian(pos,axis=1); col=np.nanmedian(pos,axis=0); glob=np.nanmedian(pos)
/tmp/ipykernel_10402/1002375550.py:26: RuntimeWarning: All-NaN slice encountered
  ref=np.nanmedian(np.where(np.asarray(adc[rows])>0,adc[rows],np.nan),axis=0)


Dead-FEE regions: 36
Dead stripes: 78
Large dead blocks: 119


### Optional manual corrections

In [6]:
# Zero-based side/module/sector/local-layer/local-pad indices.
MANUAL_DEAD_FEE = [
    # (side, module, sector, first_pad, last_pad_exclusive)
]
MANUAL_FULL_STRIPE = [
    # (side, module, sector, local_layer)
]
MANUAL_HALF_STRIPE = [
    # (side, module, sector, local_layer, "left" or "right")
]
MANUAL_DEAD_BLOCK = [
    # (side, module, sector, layer_lo, layer_hi, pad_lo, pad_hi)
]

for side,module,sector,lo,hi in MANUAL_DEAD_FEE:
    key=(side,module,sector)
    auto_masks[key]["fee"][:,lo:hi] = True
    auto_masks[key]["fee_regions"].append((lo,hi))

for side,module,sector,il in MANUAL_FULL_STRIPE:
    key=(side,module,sector)
    valid=~auto_masks[key]["fee"][il]
    auto_masks[key]["stripe_full"][il,valid]=True

for side,module,sector,il,which in MANUAL_HALF_STRIPE:
    key=(side,module,sector)
    npad=raw_maps[key]["adc"].shape[1]
    mid=npad//2
    if which=="left":
        auto_masks[key]["stripe_half"][il,:mid]=True
    elif which=="right":
        auto_masks[key]["stripe_half"][il,mid:]=True
    else:
        raise ValueError(which)

for side,module,sector,l0,l1,p0,p1 in MANUAL_DEAD_BLOCK:
    key=(side,module,sector)
    auto_masks[key]["block"][l0:l1,p0:p1]=True


## 3. Recover the underlying physical source

The reconstruction now produces two native maps per sector:

- `model_clean_maps`: FEE, blocks, single dead/hot cells, and stripes are recovered;
  this map is robustly smoothed and is used for extrapolation and primary density.
- `ibf_native_maps`: starts from the same clean physical source, then restores the
  measured dead stripes. This is the final measured IBF behavior.

A FEE repair changes only dead-like cells inside the FEE region, so a few live
channels in the middle of a damaged FEE are not overwritten. All smoothing is done
inside one native sector and therefore cannot blur across a sector boundary.


In [7]:
def local_robust_value(arr,il,ip,pad_radius,layer_radius,exclude=None):
    vals=[]
    for jl in range(max(0,il-layer_radius),min(arr.shape[0],il+layer_radius+1)):
        for jp in range(max(0,ip-pad_radius),min(arr.shape[1],ip+pad_radius+1)):
            if jl==il and jp==ip: continue
            if exclude is not None and exclude[jl,jp]: continue
            v=arr[jl,jp]
            if np.isfinite(v) and v>0: vals.append(v)
    return estimate_values(vals,"trimmed_mean") if len(vals)>=SINGLE_OUTLIER_MIN_NEIGHBORS else np.nan

def _smooth_fee_context_curve(values):
    """Fill missing layer values and robustly smooth a FEE boundary vs layer."""
    v=np.asarray(values,float).copy()
    good=np.isfinite(v)&(v>0)
    if not np.any(good): return np.full_like(v,np.nan)
    x=np.arange(len(v),dtype=float)
    if np.count_nonzero(good)==1: v[:]=v[good][0]
    else: v[:]=np.interp(x,x[good],v[good])
    size=max(1,int(FEE_CONTEXT_MEDIAN_LAYERS))
    if size%2==0: size+=1
    if size>1: v=ndimage.median_filter(v,size=size,mode="nearest")
    if FEE_CONTEXT_GAUSS_SIGMA_LAYERS>0:
        v=ndimage.gaussian_filter1d(v,float(FEE_CONTEXT_GAUSS_SIGMA_LAYERS),mode="nearest")
    return v

def _fee_context_curve(adc,columns,exclude):
    """Robust healthy-context level in each layer, then smooth it radially."""
    curve=np.full(adc.shape[0],np.nan,float)
    columns=list(columns)
    for il in range(adc.shape[0]):
        vals=[adc[il,p] for p in columns
              if not exclude[il,p] and np.isfinite(adc[il,p]) and adc[il,p]>0]
        if len(vals)>=FEE_MIN_CONTEXT_CELLS:
            curve[il]=estimate_values(vals,"trimmed_mean")
    return _smooth_fee_context_curve(curve)

def repair_fee_regions(adc,fee_mask,fee_regions,protected):
    # v13 behavior: interpolate only genuinely dead cells from healthy context
    # independently in each radial layer. This was empirically much more stable
    # than imposing an additional radial boundary model.
    out=adc.copy().astype(float)
    for lo,hi in fee_regions:
        for il,ip in zip(*np.where(fee_mask[:,lo:hi])):
            ip+=lo
            left=[out[il,p] for p in range(max(0,lo-FEE_CONTEXT_PADS),lo)
                  if not protected[il,p] and np.isfinite(out[il,p]) and out[il,p]>0]
            right=[out[il,p] for p in range(hi,min(out.shape[1],hi+FEE_CONTEXT_PADS))
                   if not protected[il,p] and np.isfinite(out[il,p]) and out[il,p]>0]
            lv=estimate_values(left,"trimmed_mean"); rv=estimate_values(right,"trimmed_mean")
            if np.isfinite(lv) and np.isfinite(rv):
                t=(ip-lo+1)/(hi-lo+1); out[il,ip]=(1-t)*lv+t*rv
            elif np.isfinite(lv): out[il,ip]=lv
            elif np.isfinite(rv): out[il,ip]=rv
    return out

def repair_mask_from_context(adc,mask,protected):
    out=adc.copy().astype(float)
    for il,ip in zip(*np.where(mask)):
        v=local_robust_value(out,il,ip,BLOCK_CONTEXT_PADS,BLOCK_CONTEXT_LAYERS,protected|mask)
        if np.isfinite(v): out[il,ip]=v
    return out

def recover_stripes(adc,stripe_mask,other_bad):
    out=adc.copy().astype(float)
    for il,ip in zip(*np.where(stripe_mask)):
        vals=[]
        for d in range(1,STRIPE_NEIGHBOR_LAYERS+1):
            for jl in (il-d,il+d):
                if 0<=jl<out.shape[0] and not other_bad[jl,ip]:
                    v=out[jl,ip]
                    if np.isfinite(v) and v>0: vals.append(v)
        if vals: out[il,ip]=estimate_values(vals,"trimmed_mean")
        else:
            v=local_robust_value(out,il,ip,2,STRIPE_NEIGHBOR_LAYERS,other_bad|stripe_mask)
            if np.isfinite(v): out[il,ip]=v
    return out

def repair_single_outliers(adc,protected):
    out=adc.copy().astype(float)
    for _ in range(SINGLE_OUTLIER_PASSES):
        new=out.copy(); changed=0
        for il in range(out.shape[0]):
            for ip in range(out.shape[1]):
                if protected[il,ip]: continue
                ref=local_robust_value(out,il,ip,SINGLE_OUTLIER_PAD_RADIUS,
                                       SINGLE_OUTLIER_LAYER_RADIUS,protected)
                if not np.isfinite(ref) or ref<=0: continue
                v=out[il,ip]
                bad=(not np.isfinite(v)) or v<=0 or v<SINGLE_DEAD_FACTOR*ref or v>SINGLE_HOT_FACTOR*ref
                if bad:
                    new[il,ip]=ref; changed+=1
        out=new
        if changed==0: break
    return out

def robust_smooth_native(adc):
    if not SMOOTH_NATIVE_MAPS: return adc.copy()
    out=adc.copy().astype(float); sm=out.copy()
    for il in range(out.shape[0]):
        for ip in range(out.shape[1]):
            vals=[]
            for jl in range(max(0,il-SMOOTH_LAYER_RADIUS),min(out.shape[0],il+SMOOTH_LAYER_RADIUS+1)):
                for jp in range(max(0,ip-SMOOTH_PAD_RADIUS),min(out.shape[1],ip+SMOOTH_PAD_RADIUS+1)):
                    v=out[jl,jp]
                    if np.isfinite(v) and v>0: vals.append(v)
            ref=estimate_values(vals,"trimmed_mean")
            if np.isfinite(ref):
                sm[il,ip]=(1.0-SMOOTH_BLEND)*out[il,ip]+SMOOTH_BLEND*ref
    return sm

model_clean_maps={}
ibf_native_maps={}
primary_stripe_native_maps={}
repair_masks={}
stripe_masks={}

for key,item in raw_maps.items():
    m=auto_masks[key]
    raw=item["adc"].astype(float)
    stripe=m["stripe"]
    other_bad=m["fee"]|m["block"]

    a=repair_fee_regions(raw,m["fee"],m["fee_regions"],stripe|m["block"])
    a=repair_mask_from_context(a,m["block"],stripe|m["fee"])
    a=recover_stripes(a,stripe,other_bad)
    if REPAIR_SINGLE_OUTLIERS: a=repair_single_outliers(a,np.zeros_like(stripe,bool))
    a=robust_smooth_native(a)

    model_clean_maps[key]=a
    ibf=a.copy()
    if KEEP_DEAD_STRIPES_IN_IBF: ibf[stripe]=raw[stripe]
    ibf_native_maps[key]=ibf

    pstripe=a.copy()
    if SAVE_PRIMARY_WITH_DEAD_STRIPES: pstripe[stripe]=raw[stripe]
    primary_stripe_native_maps[key]=pstripe

    repair_masks[key]=m["fee"]|m["block"]
    stripe_masks[key]=stripe

# Backward-compatible alias used by some QA cells.
clean_maps=ibf_native_maps

print("Underlying source recovery finished")
print("  reconstructed FEE/block/single-channel defects")
print("  dead stripes restored only in IBF / primary-with-stripes views")


Underlying source recovery finished
  reconstructed FEE/block/single-channel defects
  dead stripes restored only in IBF / primary-with-stripes views


In [8]:
# QA summary of remaining unresolved cells
for key,a in model_clean_maps.items():
    nbad=np.count_nonzero(~np.isfinite(a) | (a<=0))
    if nbad:
        side,module,sector=key
        print(f"remaining clean-map cells side={side} R{module+1} sector={sector:02d}: {nbad}")


## 4. Inspect one sector/module before trusting the global map

In [9]:
def root_graph(name,x,y,ey=None):
    x=np.asarray(x,float); y=np.asarray(y,float)
    good=np.isfinite(x)&np.isfinite(y)
    x=x[good]; y=y[good]
    g=root.TGraphErrors(len(x)) if ey is not None else root.TGraph(len(x))
    g.SetName(name)
    for i,(xx,yy) in enumerate(zip(x,y)):
        g.SetPoint(i,float(xx),float(yy))
        if ey is not None:
            ee=np.asarray(ey,float)[good][i]
            g.SetPointError(i,0.0,float(ee if np.isfinite(ee) else 0.0))
    return g

def root_th2(name,title,data,xedges,yedges):
    h=root.TH2D(name,title,len(xedges)-1,array("d",xedges),len(yedges)-1,array("d",yedges))
    h.SetDirectory(0)
    for iy in range(data.shape[0]):
        for ix in range(data.shape[1]):
            v=data[iy,ix]
            h.SetBinContent(ix+1,iy+1,float(v) if np.isfinite(v) else 0.0)
    return h

def save_canvas(c,name):
    c.SaveAs(str(QA_PLOT_DIR/f"{name}.png"))
    ROOT_QA_OBJECTS[c.GetName()]=c.Clone(c.GetName())

def style_graph(g,color,marker=20,line=1,width=3):
    g.SetLineColor(color); g.SetMarkerColor(color); g.SetMarkerStyle(marker)
    g.SetLineStyle(line); g.SetLineWidth(width); g.SetMarkerSize(1.0)

def draw_multigraph(name,title,graphs,legend_entries,xlabel,ylabel,yrange=None):
    c=root.TCanvas(name,title,1000,650); c.SetLeftMargin(0.13); c.SetBottomMargin(0.13)
    mg=root.TMultiGraph()
    leg=root.TLegend(0.58,0.68,0.89,0.89); leg.SetBorderSize(0); leg.SetFillStyle(0)
    for g,opt,label in zip(graphs,[x[0] for x in legend_entries],[x[1] for x in legend_entries]):
        mg.Add(g,opt); leg.AddEntry(g,label,"lp")
    mg.Draw("A"); mg.SetTitle(f"{title};{xlabel};{ylabel}")
    mg.GetXaxis().SetTitleSize(0.052); mg.GetYaxis().SetTitleSize(0.052)
    mg.GetXaxis().SetLabelSize(0.045); mg.GetYaxis().SetLabelSize(0.045)
    if yrange: mg.SetMinimum(yrange[0]); mg.SetMaximum(yrange[1])
    leg.Draw(); c.Modified(); c.Update()
    return c,mg,leg

def fit_powerlaw_root(name,r,y,rmin=None,rmax=None,alpha_seed=ALPHA_INITIAL,ey=None):
    r=np.asarray(r,float); y=np.asarray(y,float)
    good=np.isfinite(r)&np.isfinite(y)&(y>0)
    if ey is not None: good &= np.isfinite(ey)&(np.asarray(ey)>0)
    rr=r[good]; yy=y[good]
    if len(rr)<3: return None,None
    lo=float(np.min(rr) if rmin is None else rmin); hi=float(np.max(rr) if rmax is None else rmax)
    use=(rr>=lo)&(rr<=hi); rr=rr[use]; yy=yy[use]
    if len(rr)<3: return None,None
    e=None if ey is None else np.asarray(ey,float)[good][use]
    g=root_graph("g_"+name,rr,yy,e)
    f=root.TF1(name,f"[0]*pow({float(R_REF):.12g}/x,[1])",lo,hi)
    f.SetParNames("rho_ref","alpha")
    f.SetParameters(float(np.nanmedian(yy)),float(alpha_seed))
    f.SetParLimits(1,float(ALPHA_BOUNDS[0]),float(ALPHA_BOUNDS[1]))
    g.Fit(f,"Q0S")
    ROOT_QA_OBJECTS[g.GetName()]=g.Clone(g.GetName())
    ROOT_QA_OBJECTS[f.GetName()]=f.Clone(f.GetName())
    return g,f

def fourier_formula(n):
    terms=["[0]"]
    for k in range(1,n+1): terms += [f"[{2*k-1}]*cos({k}*x)",f"[{2*k}]*sin({k}*x)"]
    return "+".join(terms)

def fit_fourier_root(name,x,y,order,ey=None):
    x=np.asarray(x,float); y=np.asarray(y,float)
    good=np.isfinite(x)&np.isfinite(y)
    if ey is not None: good &= np.isfinite(ey)&(np.asarray(ey)>0)
    xx=x[good]; yy=y[good]
    if len(xx)<2*order+1: return None,None
    e=None if ey is None else np.asarray(ey,float)[good]
    g=root_graph("g_"+name,xx,yy,e)
    f=root.TF1(name,fourier_formula(order),PHI_MIN,PHI_MAX)
    f.SetParameter(0,float(np.nanmedian(yy)))
    g.Fit(f,"Q0S")
    ROOT_QA_OBJECTS[g.GetName()]=g.Clone(g.GetName())
    ROOT_QA_OBJECTS[f.GetName()]=f.Clone(f.GetName())
    return g,f

def array_rphi_th2(name,title,data,rvals):
    pedges=np.linspace(PHI_MIN,PHI_MAX,data.shape[1]+1)
    redges=np.empty(len(rvals)+1); redges[1:-1]=0.5*(rvals[:-1]+rvals[1:])
    redges[0]=rvals[0]-0.5*(rvals[1]-rvals[0]); redges[-1]=rvals[-1]+0.5*(rvals[-1]-rvals[-2])
    return root_th2(name,f"{title};#phi;R [cm]",data,pedges,redges)

QA_SIDE=0; QA_MODULE=2; QA_SECTOR=0; key=(QA_SIDE,QA_MODULE,QA_SECTOR)
if key in raw_maps:
    raw=raw_maps[key]["adc"]; model=model_clean_maps[key]; ibf=ibf_native_maps[key]; m=auto_masks[key]
    panels=[(raw,"Raw ADC"),(m["fee"].astype(float),"Dead FEE cells"),(m["stripe"].astype(float),"Dead stripes"),
            (model,"Recovered physical source"),(ibf,"IBF: stripes restored"),(model-raw,"Recovered - raw")]
    c=root.TCanvas("c_native_sector_qa","Native-sector QA",1800,700); c.Divide(3,2)
    keep=[]
    for i,(z,title) in enumerate(panels):
        c.cd(i+1); root.gPad.SetRightMargin(0.14)
        h=root.TH2D(f"h_native_qa_{i}",f"{title};local pad;local layer",z.shape[1],0,z.shape[1],z.shape[0],0,z.shape[0]); h.SetDirectory(0)
        for iy in range(z.shape[0]):
            for ix in range(z.shape[1]):
                v=z[iy,ix]; h.SetBinContent(ix+1,iy+1,float(v) if np.isfinite(v) else 0.0)
        h.Draw("COLZ"); keep.append(h)
    save_canvas(c,"native_sector_qa")
    print("Dead FEE:",m["fee_regions"]); print("Stripes:",m["stripe_details"]); print("Blocks:",m["block_details"])


Dead FEE: []
Stripes: [('half-left', 4, 0.9791666666666666), ('half-right', 5, 0.9895833333333334), ('half-right', 7, 0.9895833333333334)]
Blocks: [(0, 4, 0, 3, 9), (5, 15, 0, 2, 11)]


Info in <TCanvas::Print>: png file output/qa_plots/native_sector_qa.png has been created


## 5. Build measured global `(r,phi)` ADC and IBF-density maps

For each pad,

\[
A_{\rm pad}(r)\simeq r\,\Delta\phi\,\Delta r,
\qquad
\rho_{\rm IBF}\propto ADC/A_{\rm pad}.
\]

Bins with no physical pad coverage remain `NaN`, preserving sector gaps.


In [10]:
phi_edges=np.linspace(PHI_MIN,PHI_MAX,GLOBAL_PHI_BINS+1)
phi_centers=0.5*(phi_edges[:-1]+phi_edges[1:])
all_r=np.concatenate([module_layer_radii_cm(m) for m in range(N_MODULES)])
all_r=np.sort(all_r)

def phi_bin_index(phi):
    idx=np.floor((phi-PHI_MIN)/(PHI_MAX-PHI_MIN)*GLOBAL_PHI_BINS).astype(int)
    return np.clip(idx,0,GLOBAL_PHI_BINS-1)

def build_global_measured_maps(side,native_maps):
    adc_sum=np.zeros((len(all_r),GLOBAL_PHI_BINS))
    rho_sum=np.zeros_like(adc_sum)
    counts=np.zeros_like(adc_sum)
    repaired_counts=np.zeros_like(adc_sum)
    unstable_counts=np.zeros_like(adc_sum)

    rlookup={(m,il):int(np.argmin(np.abs(all_r-r)))
             for m in range(N_MODULES) for il,r in enumerate(module_layer_radii_cm(m))}

    for module in range(N_MODULES):
        for sector in range(N_SECTORS):
            key=(side,module,sector)
            if key not in native_maps: continue
            a=native_maps[key]
            repaired=repair_masks[key]
            unstable=repair_masks[key] | stripe_masks[key]

            for il in range(LAYERS_PER_MODULE):
                ir=rlookup[(module,il)]
                bins=phi_bin_index(sector_phi_centers(module,sector,il,side))
                area=pad_area_cm2(module,il,side)

                for ip,val in enumerate(a[il]):
                    if not np.isfinite(val): continue
                    ib=bins[ip]
                    adc_sum[ir,ib]+=val
                    rho_sum[ir,ib]+=val/area
                    counts[ir,ib]+=1
                    repaired_counts[ir,ib]+=float(repaired[il,ip])
                    unstable_counts[ir,ib]+=float(unstable[il,ip])

    adc=np.full_like(adc_sum,np.nan); rho=np.full_like(rho_sum,np.nan)
    repaired_fraction=np.full_like(rho_sum,np.nan)
    unstable_fraction=np.full_like(rho_sum,np.nan)
    good=counts>0
    adc[good]=adc_sum[good]/counts[good]
    rho[good]=rho_sum[good]/counts[good]
    repaired_fraction[good]=repaired_counts[good]/counts[good]
    unstable_fraction[good]=unstable_counts[good]/counts[good]
    return adc,rho,counts,repaired_fraction,unstable_fraction

adc_measured={}
ibf_measured={}
ibf_measured_clean={}
coverage_measured={}
coverage_clean={}
repair_fraction_measured={}
unstable_fraction_measured={}

raw_native_maps={key:item["adc"] for key,item in raw_maps.items()}

for side in range(N_SIDES):
    # Keep a genuinely measured ADC map for cross-run calibration: no FEE fill,
    # stripe recovery, single-cell repair, or native smoothing is applied here.
    adc_measured[side],_,coverage_measured[side],repair_fraction_measured[side],unstable_fraction_measured[side] = \
        build_global_measured_maps(side,raw_native_maps)
    _,ibf_measured[side],_,_,_ = \
        build_global_measured_maps(side,ibf_native_maps)
    _,ibf_measured_clean[side],coverage_clean[side],_,_ = \
        build_global_measured_maps(side,model_clean_maps)

print("Measured r-phi maps built")

# Explicit coverage diagnostic around +/-pi.
for side in range(N_SIDES):
    ncover=np.sum(coverage_measured[side]>0,axis=0)
    print(f"side {side}: covered phi bins = {np.count_nonzero(ncover)}/{GLOBAL_PHI_BINS}, "
          f"edge coverage (-pi,+pi) = {ncover[0]}, {ncover[-1]}")


Measured r-phi maps built
side 0: covered phi bins = 708/720, edge coverage (-pi,+pi) = 48, 48
side 1: covered phi bins = 708/720, edge coverage (-pi,+pi) = 48, 48


## 6. Inner extrapolation: robust reference density and controlled phi damping

The previous empty extrapolation had a simple cause:

```python
N_R1_AMPLITUDE_LAYERS = 8
MIN_POINTS_FOR_PHI_AMPLITUDE = 10
```

so **no phi bin could ever satisfy the minimum of 10 points**.

This version uses

```python
N_R1_AMPLITUDE_LAYERS = 16
MIN_POINTS_FOR_PHI_AMPLITUDE = 10
```

so the reference density is estimated from the full measured R1 module, after
transporting every layer back to the same reference radius using alpha(phi).

The model is

\[
\rho(r,\phi)
=
\rho_{\rm ref}(\phi)
\left(\frac{R_{\rm ref}}{r}\right)^{\alpha(\phi)}.
\]

For a fine phi bin, `rho_ref(phi)` is accepted only if at least 10 R1 layers
contribute. Because exact pad-center phi changes slightly with radius, a few fine
phi bins can still be missing even though the sector itself is healthy. Those holes
are filled **only by interpolation inside the same physical sector** before the
sector-local smoothing is applied. Physical sector gaps remain gaps.

Below the first measured R1 radius we optionally damp only the relative phi
modulation while keeping the radial power-law behavior.


In [11]:
def robust_r1_alpha_fit(rho,coverage,repair_fraction,rvals,fit_tag):
    nfit=min(N_R1_FIT_LAYERS,16,len(rvals)); rr=[]; yy=[]; ee=[]; ww=[]
    for ir in range(nfit):
        good=(coverage[ir]>0)&np.isfinite(rho[ir])&(rho[ir]>0)
        if np.count_nonzero(good)<MIN_PHI_BINS_PER_R_LAYER: continue
        vals=rho[ir,good]; logv=np.log(vals); med=np.median(vals)
        madlog=max(1.4826*np.median(np.abs(logv-np.median(logv))),0.02)
        healthy=np.mean(1.0-np.nan_to_num(repair_fraction[ir,good],nan=1.0))
        if healthy<MIN_HEALTHY_FRACTION_FOR_ALPHA: continue
        rr.append(rvals[ir]); yy.append(med); ee.append(med*max(madlog/np.sqrt(np.count_nonzero(good)),0.015)/max(np.sqrt(healthy),0.1)); ww.append(np.sqrt(healthy))
    rr=np.asarray(rr,float); yy=np.asarray(yy,float); ee=np.asarray(ee,float); ww=np.asarray(ww,float)
    if len(rr)<3:
        alpha=ALPHA_INITIAL; A=np.nanmedian(yy*(rr/R_REF)**alpha) if len(rr) else 0.0
        return alpha,A,(rr,yy,np.divide(ee,yy,out=np.zeros_like(ee),where=yy>0),ww),None
    g,f=fit_powerlaw_root(f"f_{fit_tag}_alpha_sidewide",rr,yy,float(rr.min()),float(rr.max()),ALPHA_INITIAL,ee)
    return float(f.GetParameter(1)),float(f.GetParameter(0)),(rr,yy,np.divide(ee,yy,out=np.zeros_like(ee),where=yy>0),ww),(g,f)


def circular_phi_distance(phi,center):
    return np.abs((phi-center+np.pi)%(2*np.pi)-np.pi)


def fit_alpha_in_coarse_phi_bins(rho,coverage,repair_fraction,rvals,alpha_seed,fit_tag):
    nfit=min(N_R1_FIT_LAYERS,16,len(rvals)); edges=np.linspace(PHI_MIN,PHI_MAX,ALPHA_N_PHI_BINS+1)
    centers=0.5*(edges[:-1]+edges[1:]); alpha=np.full(ALPHA_N_PHI_BINS,np.nan); quality=np.zeros(ALPHA_N_PHI_BINS); npoints=np.zeros(ALPHA_N_PHI_BINS,dtype=int)
    coarse_fits=[]
    for ib in range(ALPHA_N_PHI_BINS):
        pmask=(phi_centers>=edges[ib])&(phi_centers<(edges[ib+1] if ib<ALPHA_N_PHI_BINS-1 else edges[ib+1]+1e-12))
        rr=[]; yy=[]; ee=[]
        for ir in range(nfit):
            good=pmask&(coverage[ir]>0)&np.isfinite(rho[ir])&(rho[ir]>0)
            if np.count_nonzero(good)<2: continue
            vals=rho[ir,good]; weights=np.clip(1.0-np.nan_to_num(repair_fraction[ir,good],nan=1.0),0.05,1.0)
            med=weighted_median(vals,weights)
            if not np.isfinite(med) or med<=0: continue
            rr.append(rvals[ir]); yy.append(med); ee.append(max(med/np.sqrt(np.sum(weights)),0.01*med))
        rr=np.asarray(rr,float); yy=np.asarray(yy,float); ee=np.asarray(ee,float); npoints[ib]=len(rr)
        if len(rr)<MIN_R1_POINTS_PER_ALPHA_BIN: coarse_fits.append((None,None)); continue
        g,f=fit_powerlaw_root(f"f_{fit_tag}_alpha_coarse_bin{ib:02d}",rr,yy,float(rr.min()),float(rr.max()),alpha_seed,ee)
        coarse_fits.append((g,f))
        if f:
            alpha[ib]=f.GetParameter(1); quality[ib]=np.sum(1.0/np.maximum(ee,1e-30)**2)
    return centers,alpha,quality,npoints,coarse_fits


def periodic_interp_to_fine(coarse_phi,coarse_values):
    good=np.isfinite(coarse_values)
    if np.count_nonzero(good)<2:
        return np.full(GLOBAL_PHI_BINS,np.nan)

    x=coarse_phi[good]
    y=coarse_values[good]
    order=np.argsort(x)
    x=x[order]; y=y[order]

    xext=np.r_[x[-1]-2*np.pi,x,x[0]+2*np.pi]
    yext=np.r_[y[-1],y,y[0]]
    return np.interp(phi_centers,xext,yext)


def robust_fourier_fit_coarse(coarse_phi,values,quality,max_harmonic,fit_tag):
    good=np.isfinite(values)&np.isfinite(quality)&(quality>0)
    err=np.full_like(values,np.nan,float); err[good]=1.0/np.sqrt(quality[good]/np.nanmax(quality[good]))
    g,f=fit_fourier_root(f"f_{fit_tag}_alpha_phi_n{max_harmonic}",coarse_phi,values,max_harmonic,err)
    pred=np.full_like(phi_centers,np.nan,float)
    if f:
        pred=np.array([np.clip(f.Eval(float(x)),ALPHA_BOUNDS[0],ALPHA_BOUNDS[1]) for x in phi_centers])
        coef=np.array([f.GetParameter(i) for i in range(f.GetNpar())]); names=[f"p{i}" for i in range(f.GetNpar())]
    else:
        coef=np.array([]); names=[]
    return pred,coef,names,(g,f)


def build_phi_sector_lookup(side):
    votes=np.zeros((N_SECTORS,GLOBAL_PHI_BINS),dtype=int)
    module=0

    for sector in range(N_SECTORS):
        for il in range(min(4,LAYERS_PER_MODULE)):
            ph=sector_phi_centers(module,sector,il,side)
            bins=phi_bin_index(ph)
            for b in bins:
                votes[sector,b]+=1

    sid=np.full(GLOBAL_PHI_BINS,-1,dtype=int)
    has=np.max(votes,axis=0)>0
    sid[has]=np.argmax(votes[:,has],axis=0)
    return sid


def cyclic_order_indices(indices,n_total):
    idx=np.sort(np.asarray(indices,dtype=int))
    if len(idx)<=1:
        return idx
    gaps=np.diff(np.r_[idx,idx[0]+n_total])
    cut=np.argmax(gaps)
    return np.r_[idx[cut+1:],idx[:cut+1]]


def sector_smooth_rhoref(rhoref,sector_lookup):
    """
    Smooth rho_ref(phi) inside each physical sector only.

    Important:
      * physical sector gaps remain NaN;
      * missing fine phi bins inside a sector can be interpolated from valid
        neighbors in that same sector before smoothing.
    """
    out=np.full_like(rhoref,np.nan,dtype=float)

    win=int(RHOREF_SECTOR_MEDIAN_WINDOW)
    if win%2==0:
        win+=1

    for sec in range(N_SECTORS):
        sector_idx=np.where(sector_lookup==sec)[0]
        if len(sector_idx)==0:
            continue

        order=cyclic_order_indices(sector_idx,GLOBAL_PHI_BINS)
        vals=rhoref[order].astype(float)
        valid=np.isfinite(vals) & (vals>0)

        if np.count_nonzero(valid)==0:
            continue

        # Fill holes only inside this sector. This is needed because exact pad
        # centers do not occupy exactly the same fine phi bins in all R1 layers.
        if RHOREF_FILL_MISSING_WITHIN_SECTOR and np.count_nonzero(valid)>=2:
            x=np.arange(len(vals),dtype=float)
            vals_filled=vals.copy()
            vals_filled[~valid]=np.interp(
                x[~valid],x[valid],vals[valid]
            )
        else:
            vals_filled=vals.copy()

        finite=np.isfinite(vals_filled) & (vals_filled>0)
        if np.count_nonzero(finite)<3:
            out[order[finite]]=vals_filled[finite]
            continue

        # Interpolation above should normally make the sector fully finite.
        # If not, smooth only the contiguous finite portion.
        lv=np.log(vals_filled)
        use_win=min(win,max(3,(len(lv)//2)*2-1))
        med=ndimage.median_filter(lv,size=use_win,mode="nearest")
        smooth=ndimage.gaussian_filter1d(
            med,sigma=RHOREF_SECTOR_GAUSS_SIGMA,mode="nearest"
        )
        out[order]=np.exp(smooth)

    return out


def extract_rhoref(rho,coverage,repair_fraction,rvals,alpha_phi):
    """
    rho_ref(phi) = weighted median[
        rho(r,phi) * (r/R_REF)^alpha(phi)
    ]

    rho_ref therefore always has density units.
    """
    n_amp=min(N_R1_AMPLITUDE_LAYERS,16,len(rvals))

    if MIN_POINTS_FOR_PHI_AMPLITUDE > n_amp:
        raise ValueError(
            "MIN_POINTS_FOR_PHI_AMPLITUDE cannot exceed the number of R1 layers "
            f"used for rho_ref: min points={MIN_POINTS_FOR_PHI_AMPLITUDE}, "
            f"layers used={n_amp}"
        )
    physical_phi=np.sum(coverage[:16]>0,axis=0)>=MIN_R1_COVERAGE_LAYERS

    rhoref=np.full(rho.shape[1],np.nan)
    quality=np.zeros(rho.shape[1])
    n_used=np.zeros(rho.shape[1],dtype=int)

    for ip in range(rho.shape[1]):
        if not physical_phi[ip] or not np.isfinite(alpha_phi[ip]):
            continue

        vals=[]; weights=[]
        for ir in range(n_amp):
            y=rho[ir,ip]
            if not np.isfinite(y) or y<=0 or coverage[ir,ip]<=0:
                continue

            rep=repair_fraction[ir,ip]
            healthy=max(0.05,1.0-(rep if np.isfinite(rep) else 1.0))

            vals.append(y*(rvals[ir]/R_REF)**alpha_phi[ip])
            weights.append(healthy)

        n_used[ip]=len(vals)

        if len(vals)>=MIN_POINTS_FOR_PHI_AMPLITUDE:
            rhoref[ip]=weighted_median(vals,weights)
            quality[ip]=np.sum(weights)

    return rhoref,quality,physical_phi,n_used


def highharmonic_rhoref(rhoref,quality,fit_tag="rhoref"):
    good=np.isfinite(rhoref)&(rhoref>0)&np.isfinite(quality)&(quality>0)
    x=phi_centers[good]; y=np.log(rhoref[good]); err=1.0/np.sqrt(quality[good]/np.nanmax(quality[good]))
    g,f=fit_fourier_root(f"f_{fit_tag}_rhoref_log_n{RHOREF_HIGH_HARMONIC}",x,y,RHOREF_HIGH_HARMONIC,err)
    pred=np.full_like(rhoref,np.nan,float)
    if f:
        pred[good]=np.exp([f.Eval(float(xx)) for xx in x]); coef=np.array([f.GetParameter(i) for i in range(f.GetNpar())])
    else: coef=np.array([])
    return pred,coef,(g,f)


def modulation_scale(r, r_boundary, f_ifc):
    """
    S(r_boundary)=1 and S(IFC_RADIUS_CM)=f_ifc.
    """
    if PHI_MODULATION_DAMPING == "linear":
        if r_boundary <= IFC_RADIUS_CM:
            return 1.0
        t=(r-IFC_RADIUS_CM)/(r_boundary-IFC_RADIUS_CM)
        t=np.clip(t,0.0,1.0)
        return f_ifc + (1.0-f_ifc)*t
    raise ValueError(f"Unknown PHI_MODULATION_DAMPING={PHI_MODULATION_DAMPING}")


def build_reference_powerlaw_map(rho,rvals,alpha_phi,rhoref,
                                 phi_modulation_ifc_fraction=1.0):
    """
    Extrapolate only below first measured R1.

    rho_ref(phi) is decomposed into a robust mean and relative phi modulation.
    Only the modulation is damped inward; the mean radial profile remains.
    """
    r_boundary=rvals[0]
    extra_r=np.linspace(IFC_RADIUS_CM,r_boundary,N_INNER_EXTRAP_BINS,endpoint=False)

    out_r=np.concatenate([extra_r,rvals])
    out=np.full((len(out_r),rho.shape[1]),np.nan)

    # IMPORTANT: measured region is copied exactly, untouched.
    out[len(extra_r):]=rho

    good=np.isfinite(rhoref) & (rhoref>0) & np.isfinite(alpha_phi)
    if not np.any(good):
        return out_r,out,len(extra_r)

    # Robust phi-average reference density.
    ref_mean=np.nanmedian(rhoref[good])
    delta=np.full_like(rhoref,np.nan,dtype=float)
    delta[good]=rhoref[good]/ref_mean - 1.0

    for i,r in enumerate(extra_r):
        S=modulation_scale(r,r_boundary,phi_modulation_ifc_fraction)
        local_ref=np.full_like(rhoref,np.nan,dtype=float)
        local_ref[good]=ref_mean*(1.0 + S*delta[good])

        # Guard against pathological negative values if delta is very large.
        local_ref[good]=np.clip(local_ref[good],0.0,None)

        out[i,good]=local_ref[good]*(R_REF/r)**alpha_phi[good]

    return out_r,out,len(extra_r)


def construct_extrapolation_family(rho,coverage,repair_fraction,rvals,side,label):
    fit_tag=f"{label.lower()}_side{side}"
    alpha0,A0,fitpts,result=robust_r1_alpha_fit(rho,coverage,repair_fraction,rvals,fit_tag)

    coarse_phi,alpha_coarse_raw,alpha_quality,alpha_npoints,alpha_coarse_fits = fit_alpha_in_coarse_phi_bins(rho,coverage,repair_fraction,rvals,alpha0,fit_tag)

    alpha_coarse_fine=periodic_interp_to_fine(coarse_phi,alpha_coarse_raw)
    alpha_low,coef_low,names_low,alpha_phi_fit=robust_fourier_fit_coarse(coarse_phi,alpha_coarse_raw,alpha_quality,ALPHA_LOW_HARMONIC,fit_tag)

    physical=np.any(coverage[:16]>0,axis=0)
    alpha_constant=np.full(rho.shape[1],np.nan)
    alpha_constant[physical]=alpha0
    alpha_coarse_fine[~physical]=np.nan
    alpha_low[~physical]=np.nan

    alpha_models={
        "constant":alpha_constant,
        "coarse":alpha_coarse_fine,
        "lowharmonic":alpha_low,
    }
    final_alpha=alpha_models[ALPHA_PHI_FINAL]

    rhoref_bin,qref,_,rhoref_n_used=extract_rhoref(
        rho,coverage,repair_fraction,rvals,final_alpha
    )

    sector_lookup=build_phi_sector_lookup(side)
    rhoref_sector=sector_smooth_rhoref(rhoref_bin,sector_lookup)
    rhoref_high,coef_high,rhoref_fit=highharmonic_rhoref(rhoref_bin,qref,fit_tag)

    rhoref_models={
        "binwise":rhoref_bin,
        "sector_smooth":rhoref_sector,
        "highharmonic":rhoref_high,
    }
    final_rhoref=rhoref_models[RHOREF_PHI_FINAL]

    # Compare alpha choices while using the same rho_ref prescription.
    alpha_maps={}
    for mode,aphi in alpha_models.items():
        ref_bin,qtmp,_,_=extract_rhoref(
            rho,coverage,repair_fraction,rvals,aphi
        )
        if RHOREF_PHI_FINAL=="binwise":
            ref_use=ref_bin
        elif RHOREF_PHI_FINAL=="sector_smooth":
            ref_use=sector_smooth_rhoref(ref_bin,sector_lookup)
        else:
            ref_use=highharmonic_rhoref(ref_bin,qtmp,fit_tag+"_variant")[0]

        rfull,tmp,nextra=build_reference_powerlaw_map(
            rho,rvals,aphi,ref_use
        )
        alpha_maps[mode]=tmp

    # Compare multiplicity models while using preferred alpha(phi).
    # For this comparison, preserve 100% of phi modulation so only the A/rho_ref
    # prescription changes.
    ref_maps={}
    for mode,refphi in rhoref_models.items():
        rfull,tmp,nextra=build_reference_powerlaw_map(
            rho,rvals,final_alpha,refphi,
            phi_modulation_ifc_fraction=1.0
        )
        ref_maps[mode]=tmp

    # NEW: compare how much of the measured phi modulation survives inward.
    damping_maps={}
    damping_labels={}
    for frac in PHI_MODULATION_IFC_FRACTIONS:
        if abs(frac-1.0)<1e-9:
            tag="keep100"
        elif abs(frac-0.5)<1e-9:
            tag="keep50"
        elif abs(frac)<1e-9:
            tag="flatIFC"
        else:
            tag=f"keep{int(round(100*frac))}"

        rfull,tmp,nextra=build_reference_powerlaw_map(
            rho,rvals,final_alpha,final_rhoref,
            phi_modulation_ifc_fraction=float(frac)
        )
        damping_maps[tag]=tmp
        damping_labels[tag]=float(frac)

    # Select requested final damping fraction.
    final_tag=None
    for tag,frac in damping_labels.items():
        if abs(frac-PHI_MODULATION_IFC_FINAL)<1e-9:
            final_tag=tag
            break

    if final_tag is None:
        # Support arbitrary fraction not explicitly in the QA list.
        final_tag=f"keep{int(round(100*PHI_MODULATION_IFC_FINAL))}"
        rfull,tmp,nextra=build_reference_powerlaw_map(
            rho,rvals,final_alpha,final_rhoref,
            phi_modulation_ifc_fraction=float(PHI_MODULATION_IFC_FINAL)
        )
        damping_maps[final_tag]=tmp
        damping_labels[final_tag]=float(PHI_MODULATION_IFC_FINAL)

    return {
        "alpha0":alpha0,
        "alpha_fitpts":fitpts,
        "alpha_coarse_phi":coarse_phi,
        "alpha_coarse_raw":alpha_coarse_raw,
        "alpha_quality":alpha_quality,
        "alpha_npoints":alpha_npoints,
        "alpha_sidewide_fit":result,
        "alpha_coarse_fits":alpha_coarse_fits,
        "alpha_phi_fit":alpha_phi_fit,
        "rhoref_fit":rhoref_fit,
        "alpha_models":alpha_models,
        "alpha_maps":alpha_maps,
        "rhoref_models":rhoref_models,
        "rhoref_n_used":rhoref_n_used,
        "rhoref_maps":ref_maps,
        "damping_maps":damping_maps,
        "damping_labels":damping_labels,
        "final_damping_tag":final_tag,
        "final_alpha":final_alpha,
        "final_rhoref":final_rhoref,
        "final_map":damping_maps[final_tag],
        "sector_lookup":sector_lookup,
        "r_full":rfull,
        "n_extrap":nextra,
        "alpha_coeff_low":(coef_low,names_low),
        "rhoref_coeff_high":coef_high,
    }


R_REF = all_r[0] if R_REF_CM is None else float(R_REF_CM)

# R_REF is only the normalization radius. It does not define the extrapolation
# boundary. The extrapolation boundary is ALWAYS the first measured R1 radius.
if not ALLOW_EXTERNAL_R_REF:
    r1_max = all_r[min(15, len(all_r)-1)]
    if R_REF < all_r[0]-2.0 or R_REF > r1_max+2.0:
        raise ValueError(
            f"R_REF_CM={R_REF:.3f} cm is far outside measured R1. "
            f"Use R_REF_CM=None (recommended -> {all_r[0]:.3f} cm), "
            "or set ALLOW_EXTERNAL_R_REF=True intentionally."
        )

print(f"Reference radius for parameterization: R_REF = {R_REF:.3f} cm")
print(f"Extrapolation boundary: r < {all_r[0]:.3f} cm only")

ibf_family={}
ibf_map={}
ibf_alpha={}
ibf_alpha_fit_points={}
ibf_alpha_models={}
ibf_rhoref_variants={}
ibf_alpha_map_variants={}
ibf_rhoref_map_variants={}
ibf_damping_map_variants={}

for side in range(N_SIDES):
    fam=construct_extrapolation_family(
        ibf_measured_clean[side],coverage_clean[side],
        repair_fraction_measured[side],all_r,side,"IBF"
    )

    ibf_family[side]=fam
    ibf_map[side]=fam["final_map"].copy()
    # The extrapolated region comes from the fully recovered/smoothed physical
    # source. Only the measured region gets the actual dead stripes restored.
    ibf_map[side][fam["n_extrap"]:,:]=ibf_measured[side]
    ibf_alpha[side]=fam["alpha0"]
    ibf_alpha_fit_points[side]=fam["alpha_fitpts"]
    ibf_alpha_models[side]=fam["alpha_models"]
    ibf_rhoref_variants[side]=fam["rhoref_models"]
    ibf_alpha_map_variants[side]=fam["alpha_maps"]
    ibf_rhoref_map_variants[side]=fam["rhoref_maps"]
    ibf_damping_map_variants[side]=fam["damping_maps"]
    r_full=fam["r_full"]
    n_extrap=fam["n_extrap"]

    print(
        f"IBF side {side}: constant alpha={fam['alpha0']:.3f}; "
        f"final alpha={ALPHA_PHI_FINAL}; final rho_ref={RHOREF_PHI_FINAL}; "
        f"phi damping={fam['final_damping_tag']}"
    )

    graw=root_graph(f"g_ibf_alpha_coarse_display_side{side}",fam["alpha_coarse_phi"],fam["alpha_coarse_raw"])
    gconst=root_graph(f"g_ibf_alpha_constant_display_side{side}",phi_centers,fam["alpha_models"]["constant"])
    glow=root_graph(f"g_ibf_alpha_low_display_side{side}",phi_centers,fam["alpha_models"]["lowharmonic"])
    style_graph(graw,root.kGray+2,20,1,1); style_graph(gconst,root.kBlue+1,1,2,3); style_graph(glow,root.kRed+1,1,1,3)
    c,mg,leg=draw_multigraph(f"c_ibf_alpha_side{side}",f"IBF #alpha(#phi) — side {side}",
                             [graw,gconst,glow],[("P","coarse #alpha points"),("L","constant"),("L",f"Fourier n#leq{ALPHA_LOW_HARMONIC}")],
                             "#phi","#alpha",ALPHA_BOUNDS)
    save_canvas(c,f"ibf_alpha_phi_models_side{side}")

    gr0=root_graph(f"g_ibf_rhoref_bin_side{side}",phi_centers,fam["rhoref_models"]["binwise"])
    gr1=root_graph(f"g_ibf_rhoref_smooth_side{side}",phi_centers,fam["rhoref_models"]["sector_smooth"])
    style_graph(gr0,root.kGray+1,20,1,1); style_graph(gr1,root.kBlue+1,1,1,3)
    c,mg,leg=draw_multigraph(f"c_ibf_rhoref_side{side}",f"IBF reference density — side {side}",
                             [gr0,gr1],[("P","binwise"),("L","sector-local smooth")],"#phi",f"#rho_{{ref}} at R={R_REF:.2f} cm")
    save_canvas(c,f"ibf_rhoref_models_side{side}")

print(f"Added {n_extrap} inner radial bins down to {IFC_RADIUS_CM} cm")


Reference radius for parameterization: R_REF = 31.781 cm
Extrapolation boundary: r < 31.781 cm only
IBF side 0: constant alpha=1.330; final alpha=lowharmonic; final rho_ref=sector_smooth; phi damping=keep50
IBF side 1: constant alpha=1.682; final alpha=lowharmonic; final rho_ref=sector_smooth; phi damping=keep50
Added 24 inner radial bins down to 21.0 cm


Info in <TCanvas::Print>: png file output/qa_plots/ibf_alpha_phi_models_side0.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/ibf_rhoref_models_side0.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/ibf_alpha_phi_models_side1.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/ibf_rhoref_models_side1.png has been created


## 7. Final IBF maps and r / phi profiles

In [12]:
def fit_radial_powerlaw(rvals,yvals,name):
    """ROOT fit over the complete available radial range."""
    return fit_powerlaw_root(name,rvals,yvals,float(np.nanmin(rvals)),float(np.nanmax(rvals)),ALPHA_INITIAL)


In [13]:
def profile_from_map(values,axis):
    return np.nanmedian(values,axis=axis)

def meeting_profile_canvas(name,title,x,profiles,xlabel,ylabel,fit_power=False):
    c=root.TCanvas(name,title,1050,700); c.SetLeftMargin(0.14); c.SetBottomMargin(0.14); c.SetGrid()
    mg=root.TMultiGraph(); leg=root.TLegend(0.57,0.68,0.89,0.89); leg.SetBorderSize(0); leg.SetFillStyle(0)
    fits={}; keep=[mg,leg]
    for side,y in profiles.items():
        g=root_graph(f"g_{name}_side{side}",x,y); style_graph(g,ROOT_COLORS[side],20+side,1,3); mg.Add(g,"LP"); leg.AddEntry(g,f"side {side}","lp"); keep.append(g)
        ROOT_QA_OBJECTS[g.GetName()]=g.Clone(g.GetName())
        if fit_power:
            _,f=fit_radial_powerlaw(x,y,f"f_{name}_side{side}_fullrange")
            if f:
                f.SetLineColor(ROOT_COLORS[side]); f.SetLineWidth(4); f.SetLineStyle(2); fits[side]=f; keep.append(f)
                leg.AddEntry(f,f"side {side} fit: #alpha = {f.GetParameter(1):.2f}","l")
                print(f"{title} side {side}: full-range alpha = {f.GetParameter(1):.4f} +/- {f.GetParError(1):.4f}")
    mg.Draw("A"); mg.SetTitle(f"{title};{xlabel};{ylabel}")
    mg.GetXaxis().SetTitleSize(0.052); mg.GetYaxis().SetTitleSize(0.052); mg.GetXaxis().SetLabelSize(0.045); mg.GetYaxis().SetLabelSize(0.045)
    for f in fits.values(): f.Draw("SAME")
    leg.Draw(); c._keep=keep; c.Modified(); c.Update(); save_canvas(c,name)
    return c,mg,leg,fits

ibf_profile_r={side:profile_from_map(ibf_map[side],1) for side in range(N_SIDES)}
ibf_profile_phi={side:profile_from_map(ibf_map[side],0) for side in range(N_SIDES)}

for side in range(N_SIDES):
    h=array_rphi_th2(f"h_ibf_display_side{side}",f"IBF density shape — side {side}",ibf_map[side],r_full)
    c=root.TCanvas(f"c_ibf_density_side{side}","IBF density",1100,700); c.SetRightMargin(0.15); h.Draw("COLZ"); save_canvas(c,f"ibf_density_side{side}")

c_ibf_r,mg_ibf_r,leg_ibf_r,ibf_fullrange_fits=meeting_profile_canvas("c_ibf_radial_profile","IBF radial profile",r_full,ibf_profile_r,"R [cm]","median over #phi",True)
c_ibf_phi,mg_ibf_phi,leg_ibf_phi,_=meeting_profile_canvas("c_ibf_phi_profile","IBF #phi profile",phi_centers,ibf_profile_phi,"#phi","median over R",False)

for side in range(N_SIDES):
    delta=ibf_map[side][n_extrap:,:]-ibf_measured[side]
    finite=delta[np.isfinite(delta)]; print(f"side {side}: max |final - measured| above R1 boundary = {np.max(np.abs(finite)) if finite.size else 0.0:.6g}")


/tmp/ipykernel_10402/2087305676.py:2: RuntimeWarning: All-NaN slice encountered
  return np.nanmedian(values,axis=axis)


IBF radial profile side 0: full-range alpha = 1.4882 +/- 0.0366
IBF radial profile side 1: full-range alpha = 1.9515 +/- 0.0176
side 0: max |final - measured| above R1 boundary = 0
side 1: max |final - measured| above R1 boundary = 0


Info in <TCanvas::Print>: png file output/qa_plots/ibf_density_side0.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/ibf_density_side1.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/c_ibf_radial_profile.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/c_ibf_phi_profile.png has been created


## 8. Smooth 2D diagnostic fit in both r and phi

Periodic Fourier terms in phi are combined with low-order radial polynomials:

\[
\rho(r,\phi)=
\sum_k a_k r_n^k+
\sum_{n=1}^{N}\sum_k
\left[b_{nk}r_n^k\cos(n\phi)+c_{nk}r_n^k\sin(n\phi)\right].
\]

This is only a diagnostic representation. The repaired histogram can remain the
actual PHGarfield source.


In [14]:
def smooth_rphi_model(data,sigma_r=1.0,sigma_phi=2.0):
    good=np.isfinite(data); num=ndimage.gaussian_filter(np.where(good,data,0.0),(sigma_r,sigma_phi),mode=("nearest","wrap"))
    den=ndimage.gaussian_filter(good.astype(float),(sigma_r,sigma_phi),mode=("nearest","wrap"))
    return np.divide(num,den,out=np.zeros_like(num),where=den>1e-6)

ibf_fit={side:smooth_rphi_model(ibf_map[side]) for side in range(N_SIDES)}
ibf_fit_coeff={side:(np.array([]),[]) for side in range(N_SIDES)}
for side in range(N_SIDES):
    for arr,tag,title in [(ibf_fit[side],"smooth","IBF smooth diagnostic"),(ibf_map[side]-ibf_fit[side],"residual","IBF - smooth diagnostic")]:
        h=array_rphi_th2(f"h_ibf_{tag}_display_side{side}",f"{title} — side {side}",arr,r_full)
        c=root.TCanvas(f"c_ibf_{tag}_side{side}",title,1100,700); c.SetRightMargin(0.15); h.Draw("COLZ"); save_canvas(c,f"ibf_{tag}_side{side}")


Info in <TCanvas::Print>: png file output/qa_plots/ibf_smooth_side0.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/ibf_residual_side0.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/ibf_smooth_side1.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/ibf_residual_side1.png has been created


# Part II — primary-ionization density

The gain correction remains in native **sector × layer** coordinates before pads
are merged into `(r,phi)`.

Primary uses the same reference-radius parameterization as IBF:

\[
\rho_{\rm primary}(r,\phi)
=
\rho_{{\rm ref,primary}}(\phi)
\left(\frac{R_{\rm ref}}{r}\right)^{\alpha_{\rm primary}(\phi)}.
\]

This makes the azimuthal multiplicity term directly interpretable and prevents the
artificial large variation that appeared when plotting the old raw `A(phi)`.


In [15]:
def get_gain_hist_name(side):
    return GAIN_MAP_HIST_SIDE0 if side==0 else GAIN_MAP_HIST_SIDE1

def load_gain_histograms():
    fg=root.TFile.Open(str(GAIN_MAP_FILE),"READ")
    if not fg or fg.IsZombie():
        raise OSError(f"Cannot open gain file: {GAIN_MAP_FILE}")

    out={}
    for side in range(N_SIDES):
        name=get_gain_hist_name(side)
        h=fg.Get(name)
        if not h:
            raise KeyError(f"Missing gain histogram: {name}")
        h=h.Clone(f"{name}_clone")
        h.SetDirectory(0)
        out[side]=h

    fg.Close()
    return out

gain_hist=load_gain_histograms()

def gain_for_sector_layer(side,sector,global_tpc_layer):
    h=gain_hist[side]
    gain_layer=global_tpc_layer-GAIN_LAYER_OFFSET
    bx=h.GetXaxis().FindBin(sector+0.5)
    by=h.GetYaxis().FindBin(gain_layer+0.5)
    g=h.GetBinContent(bx,by)
    return float(g) if np.isfinite(g) and g>0 else np.nan

def build_primary_measured_from_native(side,native_maps):
    """
    Apply gain in native sector x layer coordinates BEFORE merging into r x phi.
    """
    rho_sum=np.zeros((len(all_r),GLOBAL_PHI_BINS))
    counts=np.zeros_like(rho_sum)

    rlookup={}
    for module in range(N_MODULES):
        for il,r in enumerate(module_layer_radii_cm(module)):
            rlookup[(module,il)]=int(np.argmin(np.abs(all_r-r)))

    for module in range(N_MODULES):
        for sector in range(N_SECTORS):
            key=(side,module,sector)
            if key not in native_maps:
                continue
            a=native_maps[key]

            for il in range(LAYERS_PER_MODULE):
                glayer=global_layer(module,il)
                g=gain_for_sector_layer(side,sector,glayer)
                if not np.isfinite(g):
                    continue

                ir=rlookup[(module,il)]
                phis=sector_phi_centers(module,sector,il,side)
                bins=phi_bin_index(phis)
                area=pad_area_cm2(module,il,side)

                for ip in range(a.shape[1]):
                    val=a[il,ip]
                    if not np.isfinite(val):
                        continue

                    rho=(val/g)/area if PRIMARY_DIVIDE_BY_GAIN else val/area
                    ib=bins[ip]
                    rho_sum[ir,ib]+=rho
                    counts[ir,ib]+=1

    out=np.full_like(rho_sum,np.nan)
    good=counts>0
    out[good]=rho_sum[good]/counts[good]
    return out,counts


primary_measured={}
primary_measured_with_stripes={}
primary_coverage={}

for side in range(N_SIDES):
    primary_measured[side],primary_coverage[side]=build_primary_measured_from_native(side,model_clean_maps)
    primary_measured_with_stripes[side],_=build_primary_measured_from_native(side,primary_stripe_native_maps)

primary_family={}
primary_map={}
primary_alpha={}
primary_alpha_fit_points={}
primary_alpha_models={}
primary_rhoref_variants={}
primary_alpha_map_variants={}
primary_rhoref_map_variants={}
primary_damping_map_variants={}

for side in range(N_SIDES):
    fam=construct_extrapolation_family(
        primary_measured[side],primary_coverage[side],
        repair_fraction_measured[side],all_r,side,"Primary"
    )

    primary_family[side]=fam
    primary_map[side]=fam["final_map"].copy()
    # Second primary copy: identical clean extrapolation, but measured dead
    # stripes are restored above the first measured R1 layer.
    if "primary_map_with_stripes" not in globals(): primary_map_with_stripes={}
    primary_map_with_stripes[side]=fam["final_map"].copy()
    primary_map_with_stripes[side][fam["n_extrap"]:,:]=primary_measured_with_stripes[side]
    primary_alpha[side]=fam["alpha0"]
    primary_alpha_fit_points[side]=fam["alpha_fitpts"]
    primary_alpha_models[side]=fam["alpha_models"]
    primary_rhoref_variants[side]=fam["rhoref_models"]
    primary_alpha_map_variants[side]=fam["alpha_maps"]
    primary_rhoref_map_variants[side]=fam["rhoref_maps"]
    primary_damping_map_variants[side]=fam["damping_maps"]
    r_primary=fam["r_full"]

    print(
        f"PRIMARY side {side}: constant alpha={fam['alpha0']:.3f}; "
        f"final alpha={ALPHA_PHI_FINAL}; final rho_ref={RHOREF_PHI_FINAL}; "
        f"phi damping={fam['final_damping_tag']}"
    )

    graw=root_graph(f"g_primary_alpha_coarse_display_side{side}",fam["alpha_coarse_phi"],fam["alpha_coarse_raw"])
    glow=root_graph(f"g_primary_alpha_low_display_side{side}",phi_centers,fam["alpha_models"]["lowharmonic"])
    style_graph(graw,root.kGray+2,20,1,1); style_graph(glow,root.kRed+1,1,1,3)
    c,mg,leg=draw_multigraph(f"c_primary_alpha_side{side}",f"Primary #alpha(#phi) — side {side}",
                             [graw,glow],[("P","coarse #alpha points"),("L",f"Fourier n#leq{ALPHA_LOW_HARMONIC}")],
                             "#phi","#alpha",ALPHA_BOUNDS)
    save_canvas(c,f"primary_alpha_phi_models_side{side}")

    gr0=root_graph(f"g_primary_rhoref_bin_side{side}",phi_centers,fam["rhoref_models"]["binwise"])
    gr1=root_graph(f"g_primary_rhoref_smooth_side{side}",phi_centers,fam["rhoref_models"]["sector_smooth"])
    style_graph(gr0,root.kGray+1,20,1,1); style_graph(gr1,root.kBlue+1,1,1,3)
    c,mg,leg=draw_multigraph(f"c_primary_rhoref_side{side}",f"Primary reference density — side {side}",
                             [gr0,gr1],[("P","binwise"),("L","sector-local smooth")],"#phi",f"#rho_{{ref}} at R={R_REF:.2f} cm")
    save_canvas(c,f"primary_rhoref_models_side{side}")

assert np.allclose(r_primary,r_full)


PRIMARY side 0: constant alpha=1.919; final alpha=lowharmonic; final rho_ref=sector_smooth; phi damping=keep50
PRIMARY side 1: constant alpha=1.832; final alpha=lowharmonic; final rho_ref=sector_smooth; phi damping=keep50


Info in <TCanvas::Print>: png file output/qa_plots/primary_alpha_phi_models_side0.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/primary_rhoref_models_side0.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/primary_alpha_phi_models_side1.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/primary_rhoref_models_side1.png has been created


## Primary maps, profiles, and 2D fits

In [16]:
primary_profile_r={side:np.nanmedian(primary_map[side],axis=1) for side in range(N_SIDES)}
primary_profile_phi={side:np.nanmedian(primary_map[side],axis=0) for side in range(N_SIDES)}

for side in range(N_SIDES):
    h=array_rphi_th2(f"h_primary_display_side{side}",f"Primary density shape — side {side}",primary_map[side],r_full)
    c=root.TCanvas(f"c_primary_density_side{side}","Primary density",1100,700); c.SetRightMargin(0.15); h.Draw("COLZ"); save_canvas(c,f"primary_density_side{side}")
    
c_primary_r,mg_primary_r,leg_primary_r,primary_fullrange_fits=meeting_profile_canvas("c_primary_radial_profile","Primary radial profile",r_full,primary_profile_r,"R [cm]","median over #phi",True)
c_primary_phi,mg_primary_phi,leg_primary_phi,_=meeting_profile_canvas("c_primary_phi_profile","Primary #phi profile",phi_centers,primary_profile_phi,"#phi","median over R",False)

primary_fit={side:smooth_rphi_model(primary_map[side]) for side in range(N_SIDES)}
primary_fit_coeff={side:(np.array([]),[]) for side in range(N_SIDES)}
for side in range(N_SIDES):
    for arr,tag,title in [(primary_fit[side],"smooth","Primary smooth diagnostic"),(primary_map[side]-primary_fit[side],"residual","Primary - smooth diagnostic")]:
        h=array_rphi_th2(f"h_primary_{tag}_display_side{side}",f"{title} — side {side}",arr,r_full)
        c=root.TCanvas(f"c_primary_{tag}_side{side}",title,1100,700); c.SetRightMargin(0.15); h.Draw("COLZ"); save_canvas(c,f"primary_{tag}_side{side}")


Primary radial profile side 0: full-range alpha = 1.8960 +/- 0.0092
Primary radial profile side 1: full-range alpha = 2.0462 +/- 0.0072


/tmp/ipykernel_10402/2168973059.py:2: RuntimeWarning: All-NaN slice encountered
  primary_profile_phi={side:np.nanmedian(primary_map[side],axis=0) for side in range(N_SIDES)}
Info in <TCanvas::Print>: png file output/qa_plots/primary_density_side0.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/primary_density_side1.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/c_primary_radial_profile.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/c_primary_phi_profile.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/primary_smooth_side0.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/primary_residual_side0.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/primary_smooth_side1.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/primary_residual_side1.png has been created


## Save the final maps to ROOT

In [17]:
def centers_to_edges(x):
    x=np.asarray(x,float)
    e=np.empty(len(x)+1)
    e[1:-1]=0.5*(x[:-1]+x[1:])
    e[0]=x[0]-0.5*(x[1]-x[0])
    e[-1]=x[-1]+0.5*(x[-1]-x[-2])
    return e

def numpy_to_th2(name,title,data,rvals):
    pedges=np.linspace(PHI_MIN,PHI_MAX,GLOBAL_PHI_BINS+1)
    redges=centers_to_edges(rvals)
    h=root.TH2D(
        name,title,
        GLOBAL_PHI_BINS,array("d",pedges),
        len(rvals),array("d",redges)
    )
    h.SetDirectory(0)

    for ir in range(len(rvals)):
        for ip in range(GLOBAL_PHI_BINS):
            v=data[ir,ip]
            h.SetBinContent(ip+1,ir+1,float(v) if np.isfinite(v) else 0.0)
    return h

def numpy_to_th1_centers(name,title,values,centers,xlabel,ylabel):
    edges=centers_to_edges(centers)
    h=root.TH1D(name,f"{title};{xlabel};{ylabel}",len(centers),array("d",edges))
    h.SetDirectory(0)
    for i,v in enumerate(values):
        h.SetBinContent(i+1,float(v) if np.isfinite(v) else 0.0)
    return h

def phi_profile_to_th1(name,title,values,ylabel="value"):
    edges=np.linspace(PHI_MIN,PHI_MAX,len(values)+1)
    h=root.TH1D(name,f"{title};#phi;{ylabel}",len(values),array("d",edges))
    h.SetDirectory(0)
    for i,v in enumerate(values):
        h.SetBinContent(i+1,float(v) if np.isfinite(v) else 0.0)
    return h

def fit_points_to_graph(name,fitpts):
    fr,fy,fs,fw=fitpts
    g=root.TGraphErrors(len(fr))
    g.SetName(name)
    g.SetTitle(name)
    for i,(r,y,s) in enumerate(zip(fr,fy,fs)):
        g.SetPoint(i,float(r),float(y))
        g.SetPointError(i,0.0,float(y*s))
    return g

def coarse_alpha_graph(name,phi,alpha):
    good=np.isfinite(alpha)
    g=root.TGraph(np.count_nonzero(good))
    g.SetName(name)
    g.SetTitle(name)
    j=0
    for x,y in zip(phi[good],alpha[good]):
        g.SetPoint(j,float(x),float(y))
        j+=1
    return g


def write_exposure_metadata(f):
    f.cd()
    values={
        "n_events":N_EVENTS,
        "time_requested_start_tbin":REQUESTED_TIME_START_TBIN,
        "time_start_tbin":TIME_START_TBIN,
        "time_auto_end_tbin":auto_end_tbin,
        "time_end_tbin":TIME_END_TBIN,
        "time_record_length_tbins":RECORD_LENGTH_TBINS,
        "exposure_frames_tbins":EXPOSURE_FRAMES_TBINS,
        "tpc_timebin_ns":TPC_TIMEBIN_NS,
        "exposure_frames_ns":EXPOSURE_FRAMES_NS,
    }
    for name,value in values.items():
        root.TParameter("double")(name,float(value)).Write()
    root.TNamed("time_profile_sources",", ".join(time_profile_sources)).Write()
    root.TNamed("time_window_definition",f"[{TIME_START_TBIN},{TIME_END_TBIN}) raw TPC time bins").Write()


def time_profile_th1(name,title,values):
    edges=np.r_[time_lows,time_highs[-1]]
    h=root.TH1D(name,title,len(values),array("d",edges))
    h.SetDirectory(0)
    for i,v in enumerate(values): h.SetBinContent(i+1,float(v))
    return h


# ============================================================
# Main physics maps
# ============================================================
fout=root.TFile.Open(str(OUTPUT_ROOT),"RECREATE")

for side in range(N_SIDES):
    objects=[
        # Measured, pre-gain ADC map and stability metadata are saved in the main
        # file so cross-run k_eff calibration does not need to reopen QA files.
        numpy_to_th2(
            f"h_adc_measured_rphi_side{side}",
            f"Measured exposure-normalized ADC side {side};#phi;R [cm];ADC/(frame*tbin)",
            adc_measured[side],all_r
        ),
        numpy_to_th2(
            f"h_unstable_fraction_rphi_side{side}",
            f"Fraction of pads flagged FEE/block/stripe side {side};#phi;R [cm];fraction",
            unstable_fraction_measured[side],all_r
        ),
        numpy_to_th2(
            f"h_coverage_measured_rphi_side{side}",
            f"Measured pad coverage side {side};#phi;R [cm];pads",
            coverage_measured[side],all_r
        ),
        numpy_to_th2(
            f"h_ibf_final_rphi_side{side}",
            f"Final IBF density side {side};#phi;R [cm];arb. density",
            ibf_map[side],r_full
        ),
        numpy_to_th2(
            f"h_ibf_underlying_clean_rphi_side{side}",
            f"Underlying clean IBF density side {side};#phi;R [cm];arb. density",
            ibf_family[side]["final_map"],r_full
        ),
        numpy_to_th2(
            f"h_ibf_alpha_constant_rphi_side{side}",
            f"IBF constant alpha side {side};#phi;R [cm];arb. density",
            ibf_alpha_map_variants[side]["constant"],r_full
        ),
        numpy_to_th2(
            f"h_ibf_alpha_coarse_rphi_side{side}",
            f"IBF coarse alpha(phi) side {side};#phi;R [cm];arb. density",
            ibf_alpha_map_variants[side]["coarse"],r_full
        ),
        numpy_to_th2(
            f"h_ibf_alpha_lowharmonic_rphi_side{side}",
            f"IBF low-harmonic alpha(phi) side {side};#phi;R [cm];arb. density",
            ibf_alpha_map_variants[side]["lowharmonic"],r_full
        ),
        numpy_to_th2(
            f"h_ibf_rhoref_sector_smooth_rphi_side{side}",
            f"IBF sector-smoothed rho_ref side {side};#phi;R [cm];arb. density",
            ibf_rhoref_map_variants[side]["sector_smooth"],r_full
        ),
        numpy_to_th2(
            f"h_ibf_density_fit_rphi_side{side}",
            f"Robust smooth IBF fit side {side};#phi;R [cm];arb. density",
            ibf_fit[side],r_full
        ),
        numpy_to_th2(
            f"h_primary_final_rphi_side{side}",
            f"Final primary density side {side};#phi;R [cm];arb. density",
            primary_map[side],r_full
        ),
        numpy_to_th2(
            f"h_primary_final_with_dead_stripes_rphi_side{side}",
            f"Primary density with measured dead stripes side {side};#phi;R [cm];arb. density",
            primary_map_with_stripes[side],r_full
        ),
        numpy_to_th2(
            f"h_primary_density_fit_rphi_side{side}",
            f"Robust smooth primary fit side {side};#phi;R [cm];arb. density",
            primary_fit[side],r_full
        ),
    ]

    for h in objects:
        h.Write()

write_exposure_metadata(fout)
fout.Close()


# ============================================================
# QA ROOT
# ============================================================
fqa=root.TFile.Open(str(QA_ROOT),"RECREATE")
write_exposure_metadata(fqa)

# Time-end QA: raw profile, smoothed profile, and selected window markers.
h_time=time_profile_th1("h_record_time_profile","Run time profile;raw TPC time bin;global signal",time_profile)
h_time_sm=time_profile_th1("h_record_time_profile_smooth","Smoothed run time profile;raw TPC time bin;global signal",time_profile_smooth)
h_time.Write(); h_time_sm.Write()
c_time=root.TCanvas("c_record_time_detection","Record-time detection",1100,700)
c_time.SetLeftMargin(0.13); c_time.SetBottomMargin(0.13)
h_time.SetLineWidth(2); h_time.Draw("HIST")
h_time_sm.SetLineColor(root.kRed+1); h_time_sm.SetLineWidth(3); h_time_sm.Draw("HIST SAME")
ymax=max(h_time.GetMaximum(),h_time_sm.GetMaximum())
line_start=root.TLine(TIME_START_TBIN,0,TIME_START_TBIN,ymax); line_start.SetLineStyle(2); line_start.SetLineWidth(2); line_start.Draw()
line_end=root.TLine(TIME_END_TBIN,0,TIME_END_TBIN,ymax); line_end.SetLineColor(root.kMagenta+1); line_end.SetLineStyle(2); line_end.SetLineWidth(3); line_end.Draw()
line_thr=root.TLine(time_lows[0],time_end_diag["threshold"],time_highs[-1],time_end_diag["threshold"]); line_thr.SetLineColor(root.kGreen+2); line_thr.SetLineStyle(3); line_thr.SetLineWidth(2); line_thr.Draw()
leg_time=root.TLegend(0.55,0.67,0.89,0.89); leg_time.SetBorderSize(0); leg_time.SetFillStyle(0)
leg_time.AddEntry(h_time,"raw global time profile","l"); leg_time.AddEntry(h_time_sm,"smoothed","l")
leg_time.AddEntry(line_thr,"auto noise threshold","l"); leg_time.AddEntry(line_start,f"start = {TIME_START_TBIN}","l"); leg_time.AddEntry(line_end,f"end = {TIME_END_TBIN}","l"); leg_time.Draw()
c_time._keep=[h_time,h_time_sm,line_start,line_end,line_thr,leg_time]; c_time.Write(); c_time.SaveAs(str(QA_PLOT_DIR/"record_time_detection.png"))

# Global timing acceptance used to correct the fine native TH2 maps.
for side,info in time_acceptance_global.items():
    hfrac=info["haccepted"].Clone(f"h_time_acceptance_fraction_side{side}")
    hfrac.SetDirectory(0); hfrac.SetTitle(f"Accepted/all ADC fraction side {side};#phi;R [cm];fraction")
    hfrac.Divide(info["hall"]); hfrac.Write()
    info["haccepted"].Write(f"h_time_accepted_adc_side{side}")
    info["hall"].Write(f"h_time_allstored_adc_side{side}")

for side in range(N_SIDES):
    # measured maps
    numpy_to_th2(
        f"h_ibf_measured_side{side}",
        f"Measured IBF proxy side {side};#phi;R [cm];arb.",
        ibf_measured[side],all_r
    ).Write()

    numpy_to_th2(
        f"h_primary_measured_side{side}",
        f"Measured primary proxy side {side};#phi;R [cm];arb.",
        primary_measured[side],all_r
    ).Write()

    numpy_to_th2(
        f"h_primary_measured_with_dead_stripes_side{side}",
        f"Measured primary proxy with dead stripes side {side};#phi;R [cm];arb.",
        primary_measured_with_stripes[side],all_r
    ).Write()

    numpy_to_th2(
        f"h_repair_fraction_side{side}",
        f"Repair fraction side {side};#phi;R [cm];fraction",
        repair_fraction_measured[side],all_r
    ).Write()

    numpy_to_th2(
        f"h_unstable_fraction_side{side}",
        f"FEE/block/stripe unstable fraction side {side};#phi;R [cm];fraction",
        unstable_fraction_measured[side],all_r
    ).Write()

    numpy_to_th2(
        f"h_adc_measured_side{side}",
        f"Measured exposure-normalized ADC side {side};#phi;R [cm];ADC/(frame*tbin)",
        adc_measured[side],all_r
    ).Write()

    hg=gain_hist[side].Clone(f"h_gain_sector_layer_side{side}")
    hg.SetDirectory(0)
    hg.Write()

    # alpha(phi) variants
    for mode in ("constant","coarse","lowharmonic"):
        numpy_to_th2(
            f"h_ibf_extrap_alpha_{mode}_side{side}",
            f"IBF extrapolation alpha={mode} side {side};#phi;R [cm];arb.",
            ibf_alpha_map_variants[side][mode],r_full
        ).Write()

        numpy_to_th2(
            f"h_primary_extrap_alpha_{mode}_side{side}",
            f"Primary extrapolation alpha={mode} side {side};#phi;R [cm];arb.",
            primary_alpha_map_variants[side][mode],r_full
        ).Write()

        phi_profile_to_th1(
            f"h_ibf_alpha_phi_{mode}_side{side}",
            f"IBF alpha(phi) {mode} side {side}",
            ibf_alpha_models[side][mode],
            "alpha"
        ).Write()

        phi_profile_to_th1(
            f"h_primary_alpha_phi_{mode}_side{side}",
            f"Primary alpha(phi) {mode} side {side}",
            primary_alpha_models[side][mode],
            "alpha"
        ).Write()

    # coarse alpha points themselves
    coarse_alpha_graph(
        f"g_ibf_alpha_coarse_points_side{side}",
        ibf_family[side]["alpha_coarse_phi"],
        ibf_family[side]["alpha_coarse_raw"]
    ).Write()

    coarse_alpha_graph(
        f"g_primary_alpha_coarse_points_side{side}",
        primary_family[side]["alpha_coarse_phi"],
        primary_family[side]["alpha_coarse_raw"]
    ).Write()

    phi_profile_to_th1(
        f"h_ibf_rhoref_npoints_side{side}",
        f"IBF number of R1 layers used for rho_ref side {side}",
        ibf_family[side]["rhoref_n_used"],
        "N R1 layers"
    ).Write()

    phi_profile_to_th1(
        f"h_primary_rhoref_npoints_side{side}",
        f"Primary number of R1 layers used for rho_ref side {side}",
        primary_family[side]["rhoref_n_used"],
        "N R1 layers"
    ).Write()

    # rho_ref(phi) variants
    for mode in ("binwise","sector_smooth","highharmonic"):
        numpy_to_th2(
            f"h_ibf_extrap_rhoref_{mode}_side{side}",
            f"IBF extrapolation rho_ref={mode} side {side};#phi;R [cm];arb.",
            ibf_rhoref_map_variants[side][mode],r_full
        ).Write()

        numpy_to_th2(
            f"h_primary_extrap_rhoref_{mode}_side{side}",
            f"Primary extrapolation rho_ref={mode} side {side};#phi;R [cm];arb.",
            primary_rhoref_map_variants[side][mode],r_full
        ).Write()

        phi_profile_to_th1(
            f"h_ibf_rhoref_phi_{mode}_side{side}",
            f"IBF rho_ref(phi) {mode} side {side}",
            ibf_rhoref_variants[side][mode],
            "rho_ref"
        ).Write()

        phi_profile_to_th1(
            f"h_primary_rhoref_phi_{mode}_side{side}",
            f"Primary rho_ref(phi) {mode} side {side}",
            primary_rhoref_variants[side][mode],
            "rho_ref"
        ).Write()

    numpy_to_th2(
        f"h_ibf_measured_region_difference_side{side}",
        f"Final-minus-measured IBF above R1 boundary side {side};#phi;R [cm];difference",
        ibf_map[side][n_extrap:,:]-ibf_measured[side],all_r
    ).Write()

    # inward phi-modulation damping variants
    for tag,frac in ibf_family[side]["damping_labels"].items():
        numpy_to_th2(
            f"h_ibf_extrap_damping_{tag}_side{side}",
            f"IBF phi damping {tag} side {side};#phi;R [cm];arb.",
            ibf_damping_map_variants[side][tag],r_full
        ).Write()

        numpy_to_th2(
            f"h_primary_extrap_damping_{tag}_side{side}",
            f"Primary phi damping {tag} side {side};#phi;R [cm];arb.",
            primary_damping_map_variants[side][tag],r_full
        ).Write()

    # final maps / fits / residuals
    numpy_to_th2(
        f"h_ibf_final_side{side}",
        f"Final IBF side {side};#phi;R [cm];arb.",
        ibf_map[side],r_full
    ).Write()

    numpy_to_th2(
        f"h_ibf_fit_side{side}",
        f"IBF 2D fit side {side};#phi;R [cm];arb.",
        ibf_fit[side],r_full
    ).Write()

    numpy_to_th2(
        f"h_ibf_fit_residual_side{side}",
        f"IBF fit residual side {side};#phi;R [cm];data-fit",
        ibf_map[side]-ibf_fit[side],r_full
    ).Write()

    numpy_to_th2(
        f"h_primary_final_side{side}",
        f"Final primary side {side};#phi;R [cm];arb.",
        primary_map[side],r_full
    ).Write()

    numpy_to_th2(
        f"h_primary_fit_side{side}",
        f"Primary 2D fit side {side};#phi;R [cm];arb.",
        primary_fit[side],r_full
    ).Write()

    numpy_to_th2(
        f"h_primary_fit_residual_side{side}",
        f"Primary fit residual side {side};#phi;R [cm];data-fit",
        primary_map[side]-primary_fit[side],r_full
    ).Write()

    # 1D QA profiles
    numpy_to_th1_centers(
        f"h_ibf_profile_r_side{side}",
        f"IBF radial profile side {side}",
        ibf_profile_r[side],r_full,
        "R [cm]","median over #phi"
    ).Write()

    phi_profile_to_th1(
        f"h_ibf_profile_phi_side{side}",
        f"IBF phi profile side {side}",
        ibf_profile_phi[side],
        "median density"
    ).Write()

    numpy_to_th1_centers(
        f"h_primary_profile_r_side{side}",
        f"Primary radial profile side {side}",
        primary_profile_r[side],r_full,
        "R [cm]","median over #phi"
    ).Write()

    phi_profile_to_th1(
        f"h_primary_profile_phi_side{side}",
        f"Primary phi profile side {side}",
        primary_profile_phi[side],
        "median density"
    ).Write()

    fit_points_to_graph(
        f"g_ibf_alpha_radial_fit_points_side{side}",
        ibf_alpha_fit_points[side]
    ).Write()

    fit_points_to_graph(
        f"g_primary_alpha_radial_fit_points_side{side}",
        primary_alpha_fit_points[side]
    ).Write()

    root.TParameter("double")(
        f"ibf_alpha_constant_side{side}",float(ibf_alpha[side])
    ).Write()

    root.TParameter("double")(
        f"primary_alpha_constant_side{side}",float(primary_alpha[side])
    ).Write()

    root.TParameter("double")(
        f"R_REF_CM_side{side}",float(R_REF)
    ).Write()


    # Full-range profile fits used in meeting plots
    if side in ibf_fullrange_fits: ibf_fullrange_fits[side].Write(f"f_ibf_profile_fullrange_side{side}")
    if side in primary_fullrange_fits: primary_fullrange_fits[side].Write(f"f_primary_profile_fullrange_side{side}")



# ============================================================
# Native sector QA
# ============================================================
dsec=fqa.mkdir("sector_maps")
dsec.cd()

for (side,module,sector),item in raw_maps.items():
    raw=item["adc"]
    rep=ibf_native_maps[(side,module,sector)]
    model=model_clean_maps[(side,module,sector)]
    mask=repair_masks[(side,module,sector)].astype(float)
    stripe=stripe_masks[(side,module,sector)].astype(float)

    nx=raw.shape[1]
    ny=raw.shape[0]

    faccept=item["time_acceptance"]
    for arr,tag in [
        (raw,"raw"),
        (faccept,"time_acceptance"),
        (model,"underlying_clean"),
        (rep,"ibf_with_dead_stripes"),
        (mask,"repairmask"),
        (stripe,"deadstripe_mask"),
        (model-raw,"recovery_difference")
    ]:
        name=f"h_{tag}_side{side}_sec{sector:02d}_R{module+1}"
        h=root.TH2D(name,f"{tag};local pad;local layer",nx,0,nx,ny,0,ny)
        for iy in range(ny):
            for ix in range(nx):
                v=arr[iy,ix]
                h.SetBinContent(ix+1,iy+1,float(v) if np.isfinite(v) else 0.0)
        h.Write()


# All ROOT fit objects/graphs/canvases, including every coarse-alpha radial fit.
fqa.cd()
dfits=fqa.mkdir("root_fits_and_canvases"); dfits.cd()
for name,obj in ROOT_QA_OBJECTS.items():
    if obj: obj.Write(name)

fqa.Close()

print("Wrote main maps:",OUTPUT_ROOT)
print("Wrote QA ROOT:",QA_ROOT)
print("Saved QA PNGs under:",QA_PLOT_DIR)


Wrote main maps: output/TPC_IBF_primary_maps.root
Wrote QA ROOT: output/TPC_IBF_primary_QA.root
Saved QA PNGs under: output/qa_plots


Info in <TCanvas::Print>: png file output/qa_plots/record_time_detection.png has been created


## v11 source-model logic

The output now separates detector defects from the physical source:

- **FEE / irregular dead blocks / single hot or dead channels:** reconstructed.
- **Dead stripes:** reconstructed for extrapolation and for the clean primary map,
  but restored to the measured values in the final IBF map.
- **Primary:** two copies are written:
  - `h_primary_final_rphi_side0/1` — stripes reconstructed;
  - `h_primary_final_with_dead_stripes_rphi_side0/1` — measured stripes restored.
- **IBF:** `h_ibf_final_rphi_side0/1` keeps measured dead stripes, while
  `h_ibf_underlying_clean_rphi_side0/1` is available as QA.

All robust smoothing is performed in native sector coordinates, so sector boundaries
are never averaged together. The inward extrapolation is derived only from the
fully recovered and smoothed physical source.


In [18]:
def run_all_comparisons():
    """
    Open all produced per-run ROOT files and run the cross-run comparison plots.
    This is called only by the parent notebook after every batch child finished.
    """
    if IS_BATCH_CHILD:
        return
    # ---- original comparison cell 33 ----
    pairs=[]
    for run in COMPARISON_RUNS:
        path=Path(f"output/TPC_IBF_primary_maps_QA2_{run}.root")
        if not path.exists():
            print("Skipping missing comparison file:",path); continue
        f=root.TFile.Open(str(path))
        if not f or f.IsZombie():
            print("Skipping unreadable comparison file:",path); continue
        pairs.append((run,f))
    if len(pairs)<2:
        print("Need at least two valid output files for cross-run comparison"); return
    # Always make the requested calibration reference the ratio denominator,
    # regardless of the order in COMPARISON_RUNS.
    if KEFF_REFERENCE_RUN not in [x[0] for x in pairs]:
        raise RuntimeError(f"Reference run {KEFF_REFERENCE_RUN} is missing from comparison inputs")
    ref_pair=next(x for x in pairs if x[0]==KEFF_REFERENCE_RUN)
    pairs=[ref_pair]+[x for x in pairs if x[0]!=KEFF_REFERENCE_RUN]
    runs=[x[0] for x in pairs]; files=[x[1] for x in pairs]

    def root_profile(f,name,axis):
        """
        Stable run-comparison profile, following the old v13 behavior.

        Radial: sum over all phi bins.
        Phi: sum only over the measured TPC region 31 < R < 76 cm.
        Summing many bins gives far smaller statistical/repair fluctuations than
        taking a median separately in every displayed bin.
        """
        h=f.Get(name)
        if not h: raise KeyError(f"Missing {name} in {f.GetName()}")
        uid=root.TUUID().AsString()
        if axis=="r":
            p=h.ProjectionY(f"{name}_r_{uid}",1,h.GetNbinsX(),"e")
        else:
            y1=h.GetYaxis().FindBin(31.0+1e-6)
            y2=h.GetYaxis().FindBin(76.0-1e-6)
            p=h.ProjectionX(f"{name}_phi_{uid}",y1,y2,"e")
        p.SetDirectory(0)
        return p

    def multi_run_canvas(name,title,hname,xlabel,axis,fit=False):
        c=root.TCanvas(name,title,1200,750); c.SetLeftMargin(0.13); c.SetBottomMargin(0.13); c.SetGrid()
        leg=root.TLegend(0.58,0.55,0.89,0.89); leg.SetBorderSize(0); leg.SetFillStyle(0); first=True
        keep=[]
        for irun,(f,run) in enumerate(zip(files,runs)):
            for side in range(2):
                h=root_profile(f,f"{hname}{side}",axis); color=ROOT_COLORS[irun%len(ROOT_COLORS)]
                h.SetLineColor(color); h.SetMarkerColor(color); h.SetLineWidth(3); h.SetMarkerStyle(20+side); h.SetLineStyle(1+side)
                h.SetTitle(f"{title};{xlabel};exposure-normalized density"); h.GetXaxis().SetTitleSize(0.052); h.GetYaxis().SetTitleSize(0.052)
                h.Draw("E1" if first else "E1 SAME"); first=False; keep.append(h)
                label=f"{run} s{side}"
                if fit and axis=="r":
                    ff=root.TF1(f"f_{name}_{run}_s{side}","[0]*pow(31.5/x,[1])",h.GetXaxis().GetXmin(),h.GetXaxis().GetXmax())
                    ff.SetParameters(h.GetMaximum(),1.8); ff.SetParLimits(1,ALPHA_BOUNDS[0],ALPHA_BOUNDS[1]); h.Fit(ff,"Q0S")
                    ff.SetLineColor(color); ff.SetLineStyle(3); ff.SetLineWidth(2); ff.Draw("SAME"); keep.append(ff); label+=f", #alpha={ff.GetParameter(1):.2f}"
                leg.AddEntry(h,label,"lp")
        leg.Draw(); c.Modified(); c.Update(); return c,keep

    c,_=multi_run_canvas("c_runs_ibf_r","IBF radial profiles","h_ibf_final_rphi_side","R [cm]","r",True)
    c,_=multi_run_canvas("c_runs_ibf_phi","IBF #phi profiles","h_ibf_final_rphi_side","#phi","phi",False)
    c,_=multi_run_canvas("c_runs_primary_r","Primary radial profiles","h_primary_final_rphi_side","R [cm]","r",True)
    c,_=multi_run_canvas("c_runs_primary_phi","Primary #phi profiles","h_primary_final_rphi_side","#phi","phi",False)


    # ---- original comparison cell 34 ----
    def ratio_hist(num,den,name,min_ref_frac=0.05):
        r=num.Clone(name); r.SetDirectory(0)
        vals=[abs(den.GetBinContent(i)) for i in range(1,den.GetNbinsX()+1) if den.GetBinContent(i)!=0]
        floor=min_ref_frac*np.median(vals) if vals else 0.0
        ratios=[]
        for i in range(1,r.GetNbinsX()+1):
            a=num.GetBinContent(i); b=den.GetBinContent(i)
            if abs(b)<=floor: r.SetBinContent(i,0); r.SetBinError(i,0); continue
            q=a/b; r.SetBinContent(i,q); ratios.append(q)
        med=np.median(ratios) if ratios else 1.0; mad=1.4826*np.median(np.abs(np.asarray(ratios)-med)) if ratios else 0.0
        if mad>0:
            for i in range(1,r.GetNbinsX()+1):
                q=r.GetBinContent(i)
                if q and abs(q-med)>6*mad: r.SetBinContent(i,0); r.SetBinError(i,0)
        return r

    def ratio_canvas(name,title,hname,xlabel,axis):
        c=root.TCanvas(name,title,1200,750); c.SetLeftMargin(0.13); c.SetBottomMargin(0.13); c.SetGrid()
        leg=root.TLegend(0.15,0.72,0.89,0.89); leg.SetBorderSize(0); leg.SetFillStyle(0); leg.SetNColumns(4); first=True; keep=[leg]
        for side in range(2):
            ref=root_profile(files[0],f"{hname}{side}",axis); keep.append(ref)
            for irun,(f,run) in enumerate(zip(files[1:],runs[1:]),1):
                h=root_profile(f,f"{hname}{side}",axis)
                q=ratio_hist(h,ref,f"h_ratio_{name}_{run}_s{side}"); color=ROOT_COLORS[irun%len(ROOT_COLORS)]
                q.SetLineColor(color); q.SetMarkerColor(color); q.SetLineWidth(3); q.SetMarkerStyle(20+side); q.SetLineStyle(1+side)
                q.SetTitle(f"{title};{xlabel};ratio to {runs[0]}"); q.SetMaximum(2.5); q.SetMinimum(0.5)
                q.Draw("HIST" if first else "HIST SAME"); first=False
                leg.AddEntry(q,f"{run}/{runs[0]} s{side}","l"); keep += [h,q]
        line=root.TLine(root.gPad.GetUxmin(),1.0,root.gPad.GetUxmax(),1.0); line.SetLineStyle(2); line.Draw(); keep.append(line)
        leg.Draw(); c._keep=keep; c.Modified(); c.Update(); return c

    c=ratio_canvas("c_ratio_ibf_r","IBF radial profile ratios","h_ibf_final_rphi_side","R [cm]","r")
    c=ratio_canvas("c_ratio_ibf_phi","IBF #phi profile ratios","h_ibf_final_rphi_side","#phi","phi")
    c=ratio_canvas("c_ratio_primary_r","Primary radial profile ratios","h_primary_final_rphi_side","R [cm]","r")
    save_canvas(c,f"ratio_primary_r_runs")
    c=ratio_canvas("c_ratio_primary_phi","Primary #phi profile ratios","h_primary_final_rphi_side","#phi","phi")
    save_canvas(c,f"ratio_primary_phi_runs")

    # ============================================================
    # Robust measured-ADC ratio fits -> inferred k_eff
    # ============================================================
    def _hist2(f,name):
        h=f.Get(name)
        return h if h else None

    def stable_adc_layer_fit(run_file,ref_file,run,side):
        """Fit a constant ADC(run)/ADC(ref) using only healthy measured bins."""
        hn=_hist2(run_file,f"h_adc_measured_rphi_side{side}")
        hd=_hist2(ref_file,f"h_adc_measured_rphi_side{side}")
        un=_hist2(run_file,f"h_unstable_fraction_rphi_side{side}")
        ud=_hist2(ref_file,f"h_unstable_fraction_rphi_side{side}")
        cn=_hist2(run_file,f"h_coverage_measured_rphi_side{side}")
        cd=_hist2(ref_file,f"h_coverage_measured_rphi_side{side}")
        if not all((hn,hd,un,ud,cn,cd)):
            return None

        rr=[]; yy=[]; ee=[]; nn=[]
        min_phi=max(12,int(KEFF_STABLE_MIN_PHI_FRACTION*hn.GetNbinsX()))
        for iy in range(1,hn.GetNbinsY()+1):
            rad=hn.GetYaxis().GetBinCenter(iy)
            if not (KEFF_STABLE_R_MIN_CM<=rad<=KEFF_STABLE_R_MAX_CM): continue

            # First determine healthy positive populations and their layer scales.
            cells=[]
            anum=[]; aden=[]
            for ix in range(1,hn.GetNbinsX()+1):
                if cn.GetBinContent(ix,iy)<=0 or cd.GetBinContent(ix,iy)<=0: continue
                if un.GetBinContent(ix,iy)>KEFF_STABLE_MAX_UNSTABLE_FRACTION: continue
                if ud.GetBinContent(ix,iy)>KEFF_STABLE_MAX_UNSTABLE_FRACTION: continue
                a=hn.GetBinContent(ix,iy); b=hd.GetBinContent(ix,iy)
                if not (np.isfinite(a) and np.isfinite(b) and a>0 and b>0): continue
                cells.append((a,b)); anum.append(a); aden.append(b)
            if len(cells)<min_phi: continue

            med_a=float(np.median(anum)); med_b=float(np.median(aden))
            ratios=np.asarray([a/b for a,b in cells
                               if a>=KEFF_STABLE_MIN_SIGNAL_FRACTION*med_a
                               and b>=KEFF_STABLE_MIN_SIGNAL_FRACTION*med_b],float)
            if len(ratios)<min_phi: continue

            med=float(np.median(ratios))
            sig=float(1.4826*np.median(np.abs(ratios-med)))
            scale=max(sig,0.01*abs(med),1e-9)
            ratios=ratios[np.abs(ratios-med)<=KEFF_STABLE_RATIO_CLIP_NSIGMA*scale]
            if len(ratios)<min_phi: continue

            value=float(np.median(ratios))
            spread=float(1.4826*np.median(np.abs(ratios-value)))
            err=max(spread/max(np.sqrt(len(ratios)),1.0),
                    KEFF_STABLE_REL_ERROR_FLOOR*abs(value))
            rr.append(rad); yy.append(value); ee.append(err); nn.append(len(ratios))

        rr=np.asarray(rr,float); yy=np.asarray(yy,float); ee=np.asarray(ee,float)
        if len(yy)<KEFF_STABLE_MIN_LAYERS: return None

        # One more robust layer-level rejection before the constant fit.
        med=float(np.median(yy)); sig=float(1.4826*np.median(np.abs(yy-med)))
        scale=max(sig,0.01*abs(med),1e-9)
        keep=np.abs(yy-med)<=KEFF_STABLE_LAYER_CLIP_NSIGMA*scale
        rr,yy,ee=rr[keep],yy[keep],ee[keep]
        if len(yy)<KEFF_STABLE_MIN_LAYERS: return None

        g=root.TGraphErrors(len(rr))
        g.SetName(f"g_stable_adc_ratio_run{run}_side{side}")
        for i,(r,y,e) in enumerate(zip(rr,yy,ee)):
            g.SetPoint(i,float(r),float(y)); g.SetPointError(i,0.0,float(e))
        ffit=root.TF1(f"f_stable_adc_ratio_run{run}_side{side}","[0]",
                      KEFF_STABLE_R_MIN_CM,KEFF_STABLE_R_MAX_CM)
        ffit.SetParameter(0,float(np.median(yy)))
        g.Fit(ffit,"Q0S")
        return {"ratio":float(ffit.GetParameter(0)),
                "error":float(ffit.GetParError(0)),
                "graph":g,"fit":ffit,"nlayers":len(rr)}

    ref_file=files[0]
    stable_results={}
    stable_objects=[]
    print(f"\nStable ADC-ratio k_eff calibration: reference run {KEFF_REFERENCE_RUN}, "
          f"k_eff(ref)={KEFF_REFERENCE_VALUE:.4f}")
    print(f"Stable selection: {KEFF_STABLE_R_MIN_CM:g}<R<{KEFF_STABLE_R_MAX_CM:g} cm, "
          f"unstable fraction <= {KEFF_STABLE_MAX_UNSTABLE_FRACTION:g}")

    for run,f in zip(runs,files):
        sides={}
        for side in range(2):
            res=stable_adc_layer_fit(f,ref_file,run,side)
            sides[side]=res
            if res: stable_objects += [res["graph"],res["fit"]]
        stable_results[run]=sides

    lines=["run ratio_s0 err_s0 keff_s0 ratio_s1 err_s1 keff_s1 ratio_avg err_avg keff_avg"]
    for run in runs:
        r0=stable_results[run][0]; r1=stable_results[run][1]
        vals=[x for x in (r0,r1) if x]
        if not vals:
            print(f"{run}: no stable ADC fit"); continue
        ratios=[x["ratio"] for x in vals]; errors=[x["error"] for x in vals]
        ravg=float(np.mean(ratios))
        eavg=float(np.sqrt(np.sum(np.square(errors)))/len(errors))
        kavg=KEFF_REFERENCE_VALUE*ravg
        def fmt(res):
            if not res: return (float("nan"),float("nan"),float("nan"))
            return (res["ratio"],res["error"],KEFF_REFERENCE_VALUE*res["ratio"])
        a0,e0,k0=fmt(r0); a1,e1,k1=fmt(r1)
        print(f"{run}: s0 ADC ratio={a0:.4f} +/- {e0:.4f} -> keff={k0:.4f}; "
              f"s1 ADC ratio={a1:.4f} +/- {e1:.4f} -> keff={k1:.4f}; "
              f"average ratio={ravg:.4f} -> keff={kavg:.4f}")
        lines.append(f"{run} {a0:.8g} {e0:.8g} {k0:.8g} {a1:.8g} {e1:.8g} {k1:.8g} "
                     f"{ravg:.8g} {eavg:.8g} {kavg:.8g}")

    summary_path=OUTPUT_DIR/"keff_from_stable_adc_ratio.txt"
    summary_path.write_text("\n".join(lines)+"\n")
    print("Wrote:",summary_path)

    for side in range(2):
        entries=[(run,stable_results[run][side]) for run in runs if stable_results[run][side]]
        print(f"keff_side{side}_runs={{" +
              ", ".join(f"{run}:{KEFF_REFERENCE_VALUE*res['ratio']:.6g}" for run,res in entries) + "}")

    # Compact QA: robust radial layer ratios and their constant fits.
    cstable=root.TCanvas("c_stable_adc_ratio_fits","Stable ADC ratio fits",1200,750)
    cstable.SetLeftMargin(0.13); cstable.SetBottomMargin(0.13); cstable.SetGrid()
    mgstable=root.TMultiGraph(); legstable=root.TLegend(0.15,0.70,0.89,0.89)
    legstable.SetBorderSize(0); legstable.SetFillStyle(0); legstable.SetNColumns(4)
    fit_lines=[]
    for irun,run in enumerate(runs):
        color=ROOT_COLORS[irun%len(ROOT_COLORS)]
        for side in range(2):
            res=stable_results[run][side]
            if not res: continue
            g=res["graph"]; ff=res["fit"]
            g.SetMarkerColor(color); g.SetLineColor(color); g.SetMarkerStyle(20+side); g.SetMarkerSize(0.8)
            ff.SetLineColor(color); ff.SetLineStyle(1+side); ff.SetLineWidth(2)
            mgstable.Add(g,"P"); fit_lines.append(ff)
            legstable.AddEntry(g,f"{run} s{side}: {res['ratio']:.3f}","p")
    mgstable.Draw("A")
    mgstable.SetTitle(f"Stable measured ADC ratios to {KEFF_REFERENCE_RUN};R [cm];ADC ratio")
    mgstable.GetXaxis().SetLimits(KEFF_STABLE_R_MIN_CM,KEFF_STABLE_R_MAX_CM)
    for ff in fit_lines: ff.Draw("SAME")
    legstable.Draw(); cstable._keep=[mgstable,legstable]+stable_objects+fit_lines
    cstable.Modified(); cstable.Update(); save_canvas(cstable,"stable_adc_ratio_fits")


    # ---- original comparison cell 35 ----
    gem_current_north_runs={79513:3.14,79514:2.97,79515:2.80,79516:2.70,79526:3.70,79528:3.40,79529:3.26,79507:5.20,79508:4.5,79509:4.2,79510:3.70,79511:3.40,79512:3.30,79523:6.2,81558:10.6}
    gem_current_south_runs={79513:3.29,79514:3.12,79515:2.95,79516:2.85,79526:3.79,79528:3.49,79529:3.35,79507:5.20,79508:4.5,79509:4.2,79510:3.70,79511:3.40,79512:3.30,79523:6.2,81558:10.6}

    ratios_north = {}; ratios_south = {}

    def radial_average_ratio(run_index,side,rmin=31.,rmax=76.):
        ref=root_profile(files[0],f"h_primary_final_rphi_side{side}","r")
        h=root_profile(files[run_index],f"h_primary_final_rphi_side{side}","r")
        vals=[]
        for i in range(1,h.GetNbinsX()+1):
            r=h.GetBinCenter(i); a=h.GetBinContent(i); b=ref.GetBinContent(i)
            if rmin<=r<=rmax and b>0 and a>0: vals.append(a/b)
        vals=np.sort(vals); n=int(0.1*len(vals)); vals=vals[n:len(vals)-n] if len(vals)>2*n else vals
        return float(np.mean(vals)),float(np.std(vals)/np.sqrt(len(vals)))

    graphs=[]; c=root.TCanvas("c_primary_ratio_vs_gem","Primary ratio vs GEM current",950,700); c.SetLeftMargin(0.14); c.SetBottomMargin(0.14); c.SetGrid()
    mg=root.TMultiGraph(); leg=root.TLegend(0.17,0.72,0.45,0.88); leg.SetBorderSize(0); leg.SetFillStyle(0)

    for side,currents,color,label in [(0,gem_current_south_runs,root.kBlue+1,"South, side 0"),(1,gem_current_north_runs,root.kRed+1,"North, side 1")]:
        g=root.TGraphErrors(); g.SetName(f"g_primary_ratio_vs_gem_side{side}")
        j=0
        for run in runs[1:]:
            if run not in currents: continue
            if side == 0:
                ratios_south[run] = radial_average_ratio(runs.index(run), side)
            else:
                ratios_north[run] = radial_average_ratio(runs.index(run), side)
            ir=runs.index(run); y,ey=radial_average_ratio(ir,side); x=currents[run]/currents[79513]
            g.SetPoint(j,x,y); g.SetPointError(j,0.,ey); j+=1
            print(f"{run} side {side}: I/I79513={x:.4f}, <rho/rho79513>={y:.4f} +/- {ey:.4f}")
        g.SetMarkerStyle(20+side); g.SetMarkerSize(1.4); g.SetMarkerColor(color); g.SetLineColor(color); g.SetLineWidth(3)
        mg.Add(g,"PE"); leg.AddEntry(g,label,"pe"); graphs.append(g)

    mg.Draw("A"); mg.SetTitle("Primary density vs GEM current;I_{GEM}/I_{GEM}^{79513};<#rho/#rho_{79513}>_{31<R<76 cm}")
    mg.GetXaxis().SetTitleSize(0.052); mg.GetYaxis().SetTitleSize(0.052); mg.GetXaxis().SetLabelSize(0.045); mg.GetYaxis().SetLabelSize(0.045)
    mg.GetXaxis().SetTitleOffset(1.05); mg.GetYaxis().SetTitleOffset(1.20)

    fits=[]
    for g,color,side in zip(graphs,[root.kBlue+1,root.kRed+1],[0,1]):
        f=root.TF1(f"f_primary_ratio_vs_gem_side{side}","[0]+[1]*x",0.8,1.25); f.SetLineColor(color); f.SetLineWidth(3); f.SetLineStyle(2)
        g.Fit(f,"Q0"); f.Draw("SAME"); fits.append(f)

    line=root.TLine(mg.GetXaxis().GetXmin(),1.,mg.GetXaxis().GetXmax(),1.); line.SetLineStyle(3); line.Draw()
    leg.Draw(); c._keep=[mg,leg,line]+graphs+fits; c.Modified(); c.Draw(); save_canvas(c,"primary_ratio_vs_gem_current")

    # ---- original comparison cell 36 ----
    #keffs_south_runs={79513:0.9,79514:0.85,79515:0.80,79516:0.75,79526:1.15,79528:1.05,79529:1.00,79507:1.35}
    #keffs_north_runs={79513:1.5,79514:1.45,79515:1.40,79516:1.35,79526:1.75,79528:1.65,79529:1.55,79507:1.95}
    keffs_south_runs={79513:1.5/1.5,79514:1.45/1.5,79515:1.40/1.5,79516:1.35/1.5,79526:1.75/1.5,79528:1.65/1.5,79529:1.55/1.5,79507:2.75/1.5,79508:2.25/1.5,79509:2.1/1.5,79510:1.7/1.5,79511:1.65/1.5,79512:1.60/1.5,79523:3.1/1.5,81558:5.0/1.5}
    keffs_north_runs={79513:1.5/1.5,79514:1.45/1.5,79515:1.40/1.5,79516:1.35/1.5,79526:1.75/1.5,79528:1.65/1.5,79529:1.55/1.5,79507:2.75/1.5,79508:2.25/1.5,79509:2.1/1.5,79510:1.7/1.5,79511:1.65/1.5,79512:1.60/1.5,79523:3.1/1.5,81558:5.0/1.5}

    c1=root.TCanvas("c_keff_vs_primary_ratio","k_{eff} vs primary ratio",950,700); c1.SetLeftMargin(0.14); c1.SetBottomMargin(0.14); c1.SetGrid()
    mg1=root.TMultiGraph(); leg1=root.TLegend(0.17,0.70,0.48,0.88); leg1.SetBorderSize(0); leg1.SetFillStyle(0)
    graphs1=[]; linear_fits=[]; ratios_south={}; ratios_north={}

    for side,currents,color,label in [(0,keffs_south_runs,root.kBlue+1,"South, side 0"),(1,keffs_north_runs,root.kRed+1,"North, side 1")]:
        g=root.TGraphErrors(); g.SetName(f"g_keff_vs_primary_ratio_side{side}")
        j=0
        for run in runs[0:]:
            if run not in currents: continue
            x,ex=radial_average_ratio(runs.index(run),side); y=currents[run]
            if side==0: ratios_south[run]=(x,ex)
            else: ratios_north[run]=(x,ex)
            g.SetPoint(j,x,y); g.SetPointError(j,ex,0.05); j+=1
            print(f"{run} side {side}: <rho/rho79513>={x:.4f} +/- {ex:.4f}, keff={y:.4f}")
        g.SetMarkerStyle(20+side); g.SetMarkerSize(1.4); g.SetMarkerColor(color); g.SetLineColor(color); g.SetLineWidth(3)
        mg1.Add(g,"PE"); leg1.AddEntry(g,label,"pe"); graphs1.append(g)

        xmin,xmax=min(g.GetPointX(i) for i in range(g.GetN())),max(g.GetPointX(i) for i in range(g.GetN()))
        f=root.TF1(f"f_keff_vs_primary_ratio_side{side}","[0]+[1]*x",xmin-0.02,xmax+0.02)
        f.SetLineColor(color); f.SetLineWidth(3); f.SetLineStyle(2)
        f.SetParameters(0.0,1.0);
        g.Fit(f,"Q0S"); linear_fits.append(f)
        leg1.AddEntry(f,f"keff = ({f.GetParameter(0):.2f} #pm {f.GetParError(0):.2f}) + ({f.GetParameter(1):.2f} #pm {f.GetParError(1):.2f})*x","l")
        print(f"side {side}: keff = ({f.GetParameter(0):.3f} #pm {f.GetParError(0):.3f}) + ({f.GetParameter(1):.3f} #pm {f.GetParError(1):.3f}) ratio")

    mg1.GetYaxis().SetRangeUser(0.5,1.9)
    mg1.GetYaxis().SetRangeUser(0.5,1.9)
    mg1.Draw("A")
    mg1.SetTitle("k_{eff} vs primary density ratio;<#rho/#rho_{79513}>_{31<R<76 cm};k_{eff}")
    mg1.GetXaxis().SetTitleSize(0.052); mg1.GetYaxis().SetTitleSize(0.052)
    mg1.GetXaxis().SetLabelSize(0.045); mg1.GetYaxis().SetLabelSize(0.045)  

    for f in linear_fits: f.Draw("SAME")
    leg1.Draw()

    c1._keep=[mg1,leg1]+graphs1+linear_fits
    c1.Modified(); c1.Update()
    save_canvas(c1,"primary_ratio_vs_keff")


In [19]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import os, subprocess

BATCH_MAX_WORKERS = 16

def run_batch(input_files=BATCH_INPUT_ROOTS, notebook=NOTEBOOK_FILE,
              run_comparison=RUN_COMPARISON_AFTER_BATCH, max_workers=BATCH_MAX_WORKERS):
    if IS_BATCH_CHILD:
        print("Batch child: production-only execution")
        return [], False
    if not input_files:
        print("BATCH_INPUT_ROOTS is empty")
        return [], False

    notebook=Path(notebook).resolve()
    if not notebook.exists(): raise FileNotFoundError(f"Notebook not found: {notebook}")

    def run_one(infile):
        infile=Path(infile).resolve()
        tag=infile.stem
        env=os.environ.copy()
        env["TPC_DENSITY_INPUT"]=str(infile)
        env["TPC_DENSITY_TAG"]=tag
        env["TPC_DENSITY_BATCH_CHILD"]="1"

        executed=(OUTPUT_DIR/f"executed_{tag}.ipynb").resolve()
        executed.parent.mkdir(parents=True,exist_ok=True)
        print(f"[START] {tag}: {infile}")

        cmd=["jupyter","nbconvert","--to","notebook","--execute",str(notebook),
             "--output",executed.name,"--output-dir",str(executed.parent),
             "--ExecutePreprocessor.timeout=-1"]

        result=subprocess.run(cmd,env=env,text=True,capture_output=True)
        if result.returncode:
            raise RuntimeError(
                f"{tag} failed (exit {result.returncode})\n"
                f"STDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
            )

        print(f"[DONE ] {tag}")
        return tag

    produced=[]
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures={pool.submit(run_one,f):f for f in input_files}
        for future in as_completed(futures):
            produced.append(future.result())

    print(f"Finished {len(produced)}/{len(input_files)} files")
    return produced,run_comparison

kek=run_batch()

[START] QA2_79513: /home/yoren/yumvd.Yandex.Disk/Yura/sPHENIX/tracking/current/input/qa2/QA2_79513.root
[START] QA2_79510: /home/yoren/yumvd.Yandex.Disk/Yura/sPHENIX/tracking/current/input/qa2/QA2_79510.root
[START] QA2_79508: /home/yoren/yumvd.Yandex.Disk/Yura/sPHENIX/tracking/current/input/qa2/QA2_79508.root
[START] QA2_79509: /home/yoren/yumvd.Yandex.Disk/Yura/sPHENIX/tracking/current/input/qa2/QA2_79509.root
[START] QA2_79511: /home/yoren/yumvd.Yandex.Disk/Yura/sPHENIX/tracking/current/input/qa2/QA2_79511.root
[START] QA2_79512: /home/yoren/yumvd.Yandex.Disk/Yura/sPHENIX/tracking/current/input/qa2/QA2_79512.root
[START] QA2_79515: /home/yoren/yumvd.Yandex.Disk/Yura/sPHENIX/tracking/current/input/qa2/QA2_79515.root
[START] QA2_79516: /home/yoren/yumvd.Yandex.Disk/Yura/sPHENIX/tracking/current/input/qa2/QA2_79516.root
[START] QA2_79528: /home/yoren/yumvd.Yandex.Disk/Yura/sPHENIX/tracking/current/input/qa2/QA2_79528.root
[START] QA2_79514: /home/yoren/yumvd.Yandex.Disk/Yura/sPHENIX/tr

In [20]:
produced = []
run_comparison = True

if "kek" in locals() and kek is not None:
    produced = kek[0] if len(kek) > 0 else []
    run_comparison = bool(kek[1]) if len(kek) > 1 else False

print("Finished production for:", ", ".join(produced) if produced else "<none>")
if run_comparison:
    run_all_comparisons()

Finished production for: QA2_79528, QA2_79523, QA2_79511, QA2_79513, QA2_79508, QA2_79512, QA2_79515, QA2_79510, QA2_79509, QA2_79516, QA2_79529, QA2_79514, QA2_81558

Stable ADC-ratio k_eff calibration: reference run 79513, k_eff(ref)=1.5500
Stable selection: 31<R<76 cm, unstable fraction <= 0.05
79513: s0 ADC ratio=1.0000 +/- 0.0007 -> keff=1.5500; s1 ADC ratio=1.0000 +/- 0.0007 -> keff=1.5500; average ratio=1.0000 -> keff=1.5500
79508: s0 ADC ratio=1.1156 +/- 0.0010 -> keff=1.7291; s1 ADC ratio=1.1153 +/- 0.0011 -> keff=1.7287; average ratio=1.1154 -> keff=1.7289
79509: s0 ADC ratio=1.1690 +/- 0.0010 -> keff=1.8120; s1 ADC ratio=1.1809 +/- 0.0011 -> keff=1.8303; average ratio=1.1749 -> keff=1.8212
79510: s0 ADC ratio=1.1114 +/- 0.0010 -> keff=1.7226; s1 ADC ratio=1.1317 +/- 0.0010 -> keff=1.7541; average ratio=1.1215 -> keff=1.7384
79511: s0 ADC ratio=1.0922 +/- 0.0010 -> keff=1.6929; s1 ADC ratio=1.0934 +/- 0.0010 -> keff=1.6947; average ratio=1.0928 -> keff=1.6938
79512: s0 ADC ra

Info in <TCanvas::Print>: png file output/qa_plots/ratio_primary_r_runs.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/ratio_primary_phi_runs.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/stable_adc_ratio_fits.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/primary_ratio_vs_gem_current.png has been created
Info in <TCanvas::Print>: png file output/qa_plots/primary_ratio_vs_keff.png has been created


In [21]:
# Cross-run comparison moved to run_all_comparisons() below.


In [22]:
# Cross-run comparison moved to run_all_comparisons() below.


In [23]:
# Cross-run comparison moved to run_all_comparisons() below.


In [24]:
# Cross-run comparison moved to run_all_comparisons() below.
